## 1. Setup

In [ ]:

!pip install --upgrade pip setuptools wheel --quiet
!pip install numpy pandas matplotlib seaborn scipy sympy --quiet
!pip install torch torchvision --quiet
!pip install timm scikit-learn scikit-image --quiet
!pip install thop shap lime --quiet

import os
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
!pip install -q "huggingface_hub>=0.30.0,<1.0" hf_xet


In [ ]:
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
import os
import time
import random
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.functional import softmax
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from timm import create_model
from timm.layers import DropPath          
from thop import profile
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, auc,
)

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)

if not os.path.exists("images"):
    os.makedirs("images")

QUICK_TEST_MODE = False

GLOBAL_NUM_EPOCHS = 10

GLOBAL_RANK = 32

GLOBAL_NUM_LAYERS = 2
GLOBAL_NUM_HEADS = 8

FORCE_RETRAIN = False

QUICK_TEST_MAX_TRAIN = 64
QUICK_TEST_MAX_VAL = 16
QUICK_TEST_MAX_TEST = 16
QUICK_TEST_BATCH_SIZE = 8

from torch.utils.data import Subset

def quick_subset(torch_dataset, max_n):
    if not QUICK_TEST_MODE:
        return torch_dataset
    n = min(max_n, len(torch_dataset))
    return Subset(torch_dataset, list(range(n)))

if QUICK_TEST_MODE:
    print("QUICK_TEST_MODE is ON -- using tiny data subsets to smoke-test the notebook.")
    print("Set QUICK_TEST_MODE = False above before running for real results.\n")

## 2. Dataset Registry & Selector

Add/edit dataset paths below. Every dataset must be laid out **ImageFolder-style** (one subfolder per class), e.g.:

```
/path/to/CRC7k/
    ClassA/xxx.png
    ClassB/yyy.png
    ...
```

The four new datasets below use **placeholder paths** — replace them with your actual locations before running. `num_classes` and class names are auto-detected from each dataset's folder structure at run time, so nothing else needs to change per dataset.

In [ ]:

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

DATA_ROOT = "/kaggle/input" if IS_KAGGLE else os.environ.get("DATA_ROOT", "./data")

OUTPUT_ROOT = "/kaggle/working" if IS_KAGGLE else os.environ.get("OUTPUT_ROOT", "./outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

_CRC7K_PATH = f"{DATA_ROOT}/PLACEHOLDER_PATH/CRC7k"

DATASET_CONFIGS = {
    "Kather5k": {
        "path": "/scratch/home/admin/Kather_texture_2016_image_tiles_5000",
    },
    "CRC7k": {
        "path": _CRC7K_PATH,
    },
    "NCT100k": {
        # NCT100k is used for training/validation (90/10 split internally);
        # CRC7k is held out entirely as a disjoint external test set (no NCT100k
        # samples ever appear in the CRC7k-derived test split, and vice versa).
        "train_val_path": f"{DATA_ROOT}/PLACEHOLDER_PATH/NCT100k",
        "test_path": _CRC7K_PATH,
    },
    "LC25000": {

        "train_val_path": f"{DATA_ROOT}/datasets/javaidahmadwani/lc25000/lung_colon_image_set/Train and Validation Set"
                           if IS_KAGGLE else f"{DATA_ROOT}/LC25000/Train and Validation Set",
        "test_path": f"{DATA_ROOT}/datasets/javaidahmadwani/lc25000/lung_colon_image_set/Test Set"
                     if IS_KAGGLE else f"{DATA_ROOT}/LC25000/Test Set",
    },
    "BreakHis": {
        "path": "/scratch/home/admin/BreaKHis_v1/BreaKHis_v1/histology_slides/breast_cache/",
    },
}

def _check_dataset_paths(cfg):
    paths = [cfg["path"]] if "path" in cfg else [cfg["train_val_path"], cfg["test_path"]]
    return [p for p in paths if not os.path.isdir(p)]

SELECTED_DATASET = "BreakHis"

assert SELECTED_DATASET == "ALL" or SELECTED_DATASET in DATASET_CONFIGS, (
    f"Unknown SELECTED_DATASET '{SELECTED_DATASET}'. "
    f"Choose one of {list(DATASET_CONFIGS)} or 'ALL'."
)
DATASETS_TO_RUN = list(DATASET_CONFIGS.keys()) if SELECTED_DATASET == "ALL" else [SELECTED_DATASET]

print(f"Will run the pipeline for: {DATASETS_TO_RUN}")


In [ ]:


import re
from sklearn.model_selection import GroupShuffleSplit

def try_extract_patient_id(filename):
    patterns = [
        r'SOB_[A-Z]_[A-Z]+-(\d+-\d+)',
        r'^(P\d+)',
        r'patient[_-]?(\d+)',
        r'case[_-]?(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, filename, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return None

def get_patient_groups(image_folder_dataset):
    filepaths = [s[0] for s in image_folder_dataset.samples]
    patient_ids = [try_extract_patient_id(os.path.basename(fp)) for fp in filepaths]
    if any(p is None for p in patient_ids) or len(set(patient_ids)) < 2:
        return None
    return np.array(patient_ids)

def patient_group_split(image_folder_dataset, seed, test_frac=0.2, val_frac_of_trainval=0.1):
    groups = get_patient_groups(image_folder_dataset)
    if groups is None:
        return None

    idx = np.arange(len(image_folder_dataset))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(idx, groups=groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_trainval, random_state=seed)
    tr_sub, val_sub = next(gss2.split(trainval_idx, groups=groups[trainval_idx]))
    train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]

    assert not (set(groups[train_idx]) & set(groups[test_idx]))
    assert not (set(groups[train_idx]) & set(groups[val_idx]))
    assert not (set(groups[val_idx]) & set(groups[test_idx]))
    return train_idx, val_idx, test_idx


## 3. Canonical class & function definitions (dataset-independent, defined once)

CNN feature extractors, the LoRaS-CT / MHA-Net / attention-baseline model classes, shared training & eval utilities, GradCAM, ablation-only model variants, and the (fixed) pathology foundation-model loaders. None of these reference a specific dataset -- `num_classes` etc. are passed in as arguments at call time inside `run_pipeline` below.

In [ ]:

train_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

batch_size = QUICK_TEST_BATCH_SIZE if QUICK_TEST_MODE else 32


### 3.1 Teacher-model imports note
Teacher ViT/DeiT/Swin models are now loaded **inside** `run_pipeline` (their classification head size depends on `num_classes`, which differs per dataset).

In [ ]:
class ResNet18_Features(nn.Module):
    def __init__(self):
        super(ResNet18_Features, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])

    def forward(self, x):
        return self.features(x)

class DenseNet121_Features(nn.Module):
    def __init__(self):
        super(DenseNet121_Features, self).__init__()
        densenet = models.densenet121(pretrained=True)
        self.features = densenet.features

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=False)
        return x

In [ ]:
class LowRankSparseMultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert rank % num_heads == 0, "rank must be divisible by num_heads (rank-space multi-head split)"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.rank_head_dim = rank // num_heads
        self.sparsity_ratio = sparsity_ratio

        self.q_low = nn.Linear(embed_dim, rank, bias=False)
        self.k_low = nn.Linear(embed_dim, rank, bias=False)
        self.v_low = nn.Linear(embed_dim, rank, bias=False)

        self.rank_mix = nn.Linear(rank, rank, bias=False)

        self.out_proj = nn.Linear(rank, embed_dim, bias=False)

        self.scale = rank ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores
        num_to_keep = max(1, int(sparsity_ratio * seq_length))
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        return attn_scores.masked_fill(~sparse_mask, float('-inf'))

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()

        Q = self.q_low(x)
        K = self.k_low(x)
        V = self.v_low(x)

        Q = Q.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        sparse_attn_scores = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)

        attn_output = torch.matmul(attn_probs, V)

        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.rank)

        attn_output = self.rank_mix(attn_output)

        return self.out_proj(attn_output)

class CustomDeiTLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

class HybridStudentModel(nn.Module):
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, drop_path_rate=0.1, sparsity_ratio=0.5, grid_size=3):
        super(HybridStudentModel, self).__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_features=False):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)

        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))

        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)

        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        logits = self.classifier(pooled)
        if return_features:
            return logits, pooled
        return logits

In [ ]:
class CustomDeiTLayer_MHA(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5
        self.drop_path = nn.Identity() if drop_path == 0 else nn.Dropout(drop_path)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        normed = self.norm1(x)
        B, N, E = normed.shape
        Q = self.q_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        attn_out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        attn_out = self.out_proj(attn_out)
        x = x + self.drop_path(attn_out)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

class MHANetBaseline(nn.Module):
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 drop_path_rate=0.1, grid_size=3):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        concat_channels = 512 + 1024
        self.grid_size = grid_size

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer_MHA(embed_dim, num_heads, drop_path=drop_path_rate)
            for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)

        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))

        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x

In [ ]:

class LinformerAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, seq_len, proj_k=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.seq_len = seq_len
        self.proj_k = proj_k or max(1, seq_len // 2)

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.E_proj = nn.Linear(seq_len, self.proj_k, bias=False)
        self.F_proj = nn.Linear(seq_len, self.proj_k, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        assert N == self.seq_len, f"LinformerAttention was built for N={self.seq_len}, got {N}"
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.E_proj(self.k_proj(x).transpose(1, 2)).transpose(1, 2)
        V = self.F_proj(self.v_proj(x).transpose(1, 2)).transpose(1, 2)
        K = K.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)

class PerformerAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, nb_features=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.nb_features = nb_features or self.head_dim

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.register_buffer(
            'random_matrix', torch.randn(self.num_heads, self.head_dim, self.nb_features)
        )

    def _phi(self, x):
        proj = torch.einsum('bhnd,hdf->bhnf', x, self.random_matrix)
        return F.elu(proj) + 1

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        Qp, Kp = self._phi(Q), self._phi(K)
        KV = torch.einsum('bhnf,bhnd->bhfd', Kp, V)
        denom = torch.einsum('bhnf,bhf->bhn', Qp, Kp.sum(dim=2)) + 1e-6
        out = torch.einsum('bhnf,bhfd->bhnd', Qp, KV) / denom.unsqueeze(-1)
        out = out.transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)

class BigBirdAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size=64):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.block_size = block_size

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)

class StandardMHAWrapper(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)

class GenericAttnLayer(nn.Module):
    def __init__(self, attn_module, embed_dim, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = attn_module
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden), nn.GELU(), nn.Linear(mlp_hidden, embed_dim)
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

class GenericHybridModel(nn.Module):
    def __init__(self, num_classes, attn_factory, embed_dim=768, num_layers=GLOBAL_NUM_LAYERS,
                 grid_size=3, drop_path_rate=0.1):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024

        self.layers = nn.ModuleList([
            GenericAttnLayer(attn_factory(), embed_dim, drop_path=drop_path_rate)
            for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        r = self.resnet(x)
        d = self.densenet(x)
        feats = torch.cat((r, d), dim=1)
        if self.grid_size != feats.shape[-1]:
            feats = F.adaptive_avg_pool2d(feats, (self.grid_size, self.grid_size))
        b, c, h, w = feats.shape
        feats = feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(feats)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=3.0):
        super(DistillationLoss, self).__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, ground_truth):
        hard_loss = self.ce_loss(student_logits, ground_truth)
        soft_loss = self.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            F.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

def evaluate_model(model, data_loader, criterion, return_predictions=False):
    model.eval()
    correct, total, test_loss = 0, 0, 0.0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            if return_predictions:
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    avg_loss = test_loss / len(data_loader)
    if return_predictions:
        return accuracy, avg_loss, np.array(all_labels), np.array(all_preds)
    return accuracy, avg_loss

def train_model_with_distillation(student_model, teacher_models, train_loader, val_loader,
                                   distillation_criterion, optimizer, num_epochs=1):
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []
    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            student_outputs = student_model(images)
            with torch.no_grad():
                teacher_logits = [teacher(images) for teacher in teacher_models]
                combined_teacher_logits = sum(teacher_logits) / len(teacher_logits)
            loss = distillation_criterion(student_outputs, combined_teacher_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)
        train_accuracy, _ = evaluate_model(student_model, train_loader, distillation_criterion.ce_loss)
        val_accuracy, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Train Acc: {train_accuracy:.2f}%, Val Acc: {val_accuracy:.2f}%')
    return student_model, train_losses, val_losses, train_accuracies, val_accuracies

def train_model_plain(model, train_loader, val_loader, criterion, optimizer, num_epochs=20):
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)
        train_accuracy, _ = evaluate_model(model, train_loader, criterion)
        val_accuracy, val_loss = evaluate_model(model, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")
    return train_losses, val_losses, train_accuracies, val_accuracies

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_custom_parameters(model, exclude_list=None):
    exclude_list = exclude_list or []
    excluded_params = set(p for submodule in exclude_list for p in submodule.parameters())
    return sum(p.numel() for p in model.parameters() if p.requires_grad and p not in excluded_params)

def measure_inference_time(model, dummy_input, n_runs=50, device='cuda'):
    model.eval()
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    end = time.time()
    return (end - start) / n_runs * 1000

def measure_memory(model, dummy_input, device='cuda'):
    if device != 'cuda':
        return None
    torch.cuda.reset_peak_memory_stats(device)
    model.eval()
    with torch.no_grad():
        _ = model(dummy_input)
    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)

def measure_inference_time_and_memory(model, data_loader, device='cuda'):
    
    model.eval()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()

    start_time = time.time()
    n_images = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            n_images += images.size(0)
            outputs = model(images)
    if device == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()

    total_time = end_time - start_time
    avg_time_per_image_ms = (total_time / n_images) * 1000
    peak_memory = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == 'cuda' else None

    return peak_memory, total_time, avg_time_per_image_ms

def measure_efficiency_isolated(models_dict, dummy_input, device='cuda', n_runs=50, warmup=5):
    results = {}
    original_devices = {name: next(m.parameters()).device for name, m in models_dict.items()}

    for name, model in models_dict.items():

        for other_name, other_model in models_dict.items():
            if other_name != name:
                other_model.to('cpu')
        torch.cuda.empty_cache()
        gc.collect()

        model.to(device).eval()

        with torch.no_grad():
            for _ in range(warmup):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()

        if device == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == 'cuda' else None

        if device == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            for _ in range(n_runs):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        inf_time = (time.time() - start) / n_runs * 1000

        results[name] = {
            'Inference Time (ms)': round(inf_time, 4),
            'Peak Memory (MB)': round(peak_mem, 4) if peak_mem is not None else None,
        }
        print(f"[{name}] measured in isolation -> Time: {inf_time:.4f} ms, "
              f"Memory: {peak_mem:.2f} MB" if peak_mem is not None else
              f"[{name}] measured in isolation -> Time: {inf_time:.4f} ms")

    for name, model in models_dict.items():
        model.to(original_devices[name])

    return pd.DataFrame(results).T


In [ ]:

_THOP_ARTIFACT_SUFFIXES = ('.total_ops', '.total_params')
_THOP_ARTIFACT_NAMES = ('total_ops', 'total_params')

def _strip_thop_artifacts(state_dict):
    return {
        k: v for k, v in state_dict.items()
        if k not in _THOP_ARTIFACT_NAMES and not k.endswith(_THOP_ARTIFACT_SUFFIXES)
    }

def checkpoint_path(current_dataset, model_name):
    safe_name = (model_name.replace(" ", "_")
                 .replace("(", "").replace(")", "")
                 .replace("/", "-").replace("\\", "-")
                 .replace(":", "-"))
    return f"{OUTPUT_ROOT}/{current_dataset}_{safe_name}_checkpoint.pth"

def save_model_checkpoint(current_dataset, model_name, model, result_dict, history):
    path = checkpoint_path(current_dataset, model_name)
    torch.save({
        'model_state_dict': _strip_thop_artifacts(model.state_dict()),
        'result': result_dict,
        'history': history,
    }, path)
    print(f"  [checkpoint saved] {path}")

def load_model_checkpoint(current_dataset, model_name, model):
    path = checkpoint_path(current_dataset, model_name)
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(_strip_thop_artifacts(ckpt['model_state_dict']))
    print(f"  [checkpoint found] Skipping training for '{model_name}' -- loaded from {path}")
    return ckpt['result'], ckpt['history']


In [ ]:
class LowRankSparseMultiheadAttention_Visualizable(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention_Visualizable, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.sparsity_ratio = sparsity_ratio

        self.q_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.q_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])
        self.k_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.k_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])
        self.v_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.v_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])

        self.out_proj = nn.Linear(embed_dim * num_heads, embed_dim)
        self.scale = embed_dim ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores, torch.ones_like(attn_scores)
        num_to_keep = int(sparsity_ratio * seq_length)
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        sparse_attn_scores = attn_scores.masked_fill(~sparse_mask, float('-inf'))
        return sparse_attn_scores, sparse_mask

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()

        q, k, v = [], [], []
        for i in range(self.num_heads):
            q.append(self.q_highs[i](self.q_lows[i](x)).view(batch_size, seq_length, embed_dim))
            k.append(self.k_highs[i](self.k_lows[i](x)).view(batch_size, seq_length, embed_dim))
            v.append(self.v_highs[i](self.v_lows[i](x)).view(batch_size, seq_length, embed_dim))

        q = torch.stack(q, dim=1)
        k = torch.stack(k, dim=1)
        v = torch.stack(v, dim=1)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        sparse_attn_scores, sparse_mask = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)
        attn_output = torch.matmul(attn_probs, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.num_heads * embed_dim)

        output = self.out_proj(attn_output)
        return output, sparse_attn_scores, sparse_mask

class CustomDeiTLayer_Visualizable(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer_Visualizable, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention_Visualizable(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        attn_output, attn_scores, sparse_mask = self.attn(self.norm1(x))
        x = x + self.drop_path(attn_output)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x, attn_scores, sparse_mask

class HybridStudentModel_Visualizable(nn.Module):
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS, rank=GLOBAL_RANK,
                 drop_path_rate=0.1, sparsity_ratio=0.5):
        super(HybridStudentModel_Visualizable, self).__init__()
        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer_Visualizable(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                                          sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(2000, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.deit_embed(x)
        for layer in self.deit_layers:
            x, attn_scores, sparse_mask = layer(x)
        x = self.norm(x)
        x = x.squeeze(1)
        x = self.classifier(x)
        return x, attn_scores, sparse_mask

def visualize_attention_and_sparsity(attn_scores, sparse_mask, head=0):
    attn_scores = attn_scores[0, head].cpu().detach().numpy()
    sparse_mask = sparse_mask[0, head].cpu().detach().numpy()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5), dpi=150)
    ax1.imshow(attn_scores, cmap='viridis')
    ax1.set_title(f'Attention Scores (Head {head})')
    ax1.set_xlabel('Key Position')
    ax1.set_ylabel('Query Position')

    ax2.imshow(sparse_mask, cmap='gray')
    ax2.set_title(f'Sparsity Mask (Head {head})')
    ax2.set_xlabel('Key Position')
    ax2.set_ylabel('Query Position')
    plt.show()

def forward_with_visualization(model, data_loader, head=1):
    model.eval()
    with torch.no_grad():
        for batch in data_loader:
            images = batch.cuda() if isinstance(batch, torch.Tensor) else batch[0].cuda()
            outputs, attn_scores, sparse_mask = model(images)
            visualize_attention_and_sparsity(attn_scores, sparse_mask, head=head)
            break

demo_batch_size = 32
demo_embed_dim = 768
demo_num_classes = 8

demo_hybrid_model = HybridStudentModel_Visualizable(num_classes=demo_num_classes, embed_dim=demo_embed_dim).cuda()

demo_loader_1 = torch.utils.data.DataLoader(torch.randn(demo_batch_size, 1, 2000), batch_size=demo_batch_size)
forward_with_visualization(demo_hybrid_model, demo_loader_1, head=1)

demo_seq_len = 16
demo_loader_2 = torch.utils.data.DataLoader(torch.randn(demo_batch_size, demo_seq_len, 2000), batch_size=demo_batch_size)
forward_with_visualization(demo_hybrid_model, demo_loader_2, head=1)

In [ ]:

from scipy import stats

N_SEEDS = 1 if QUICK_TEST_MODE else 3
SEED_LIST = [42, 123, 2024][:N_SEEDS]
STAT_NUM_EPOCHS = GLOBAL_NUM_EPOCHS

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_split(dataset_path, seed):
    full_ds = datasets.ImageFolder(dataset_path)
    group_split = patient_group_split(full_ds, seed=seed)

    if group_split is not None:
        train_idx, val_idx, test_idx = group_split
        train_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
        val_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
        test_ds = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
    else:
        g = torch.Generator().manual_seed(seed)
        n_test = int(len(full_ds) * 0.2)
        n_train = len(full_ds) - n_test
        train_ds, test_ds = random_split(full_ds, [n_train, n_test], generator=g)
        n_val = int(n_train * 0.1)
        n_train2 = n_train - n_val
        train_ds, val_ds = random_split(train_ds, [n_train2, n_val], generator=g)
        train_ds.dataset.transform = train_val_transform
        val_ds.dataset.transform = train_val_transform
        test_ds.dataset.transform = test_transform

    train_ds = quick_subset(train_ds, QUICK_TEST_MAX_TRAIN)
    val_ds = quick_subset(val_ds, QUICK_TEST_MAX_VAL)
    test_ds = quick_subset(test_ds, QUICK_TEST_MAX_TEST)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    vl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    tel = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return tl, vl, tel


def _flatten_subset_to_paths(ds):
    if isinstance(ds, Subset):
        inner = ds.dataset
        if isinstance(inner, Subset):
            inner_paths = _flatten_subset_to_paths(inner)
            return [inner_paths[i] for i in ds.indices]
        else:
            return [inner.samples[i][0] for i in ds.indices]
    else:
        return [p for p, _ in ds.samples]


def _rebuild_subset_from_paths(root_path, paths, transform):
    base = datasets.ImageFolder(root_path, transform=transform)
    path_to_idx = {p: i for i, (p, _) in enumerate(base.samples)}
    idx, missing = [], []
    for p in paths:
        if p in path_to_idx:
            idx.append(path_to_idx[p])
        else:
            missing.append(p)
    if missing:
        raise RuntimeError(
            f"Split cache references {len(missing)} file(s) no longer present under '{root_path}' "
            f"(e.g. {missing[0]}). Delete the split cache file to regenerate, or restore the missing files."
        )
    return Subset(base, idx)


In [ ]:


import sympy as sp

E, N, H, R = sp.symbols('E N H R', positive=True, integer=True)

mha_params  = 4*E**2 + 4*E
mha_flops   = 4*N*E**2 + 2*N**2*E

lora_params = 4*E*R + R**2
lora_flops  = 4*N*E*R + 2*N**2*R + N*R**2

print("Standard MHA   params:", mha_params, "   FLOPs:", mha_flops)
print("LoRa-SMHA      params:", lora_params, "   FLOPs:", lora_flops)

reduction_params = sp.simplify(mha_params / lora_params)
print("\nSymbolic params ratio (MHA / LoRa-SMHA):", reduction_params)

E_val, H_val, R_val = 768, GLOBAL_NUM_HEADS, GLOBAL_RANK
rows = []
for N_val in [9, 16, 25, 49, 100]:
    mp = int(mha_params.subs({E: E_val}))
    mf = int(mha_flops.subs({E: E_val, N: N_val}))
    lp = int(lora_params.subs({E: E_val, R: R_val}))
    lf = int(lora_flops.subs({E: E_val, N: N_val, R: R_val}))
    rows.append({
        'N (tokens)': N_val, 'MHA Params': mp, 'MHA FLOPs': mf,
        'LoRa-SMHA Params': lp, 'LoRa-SMHA FLOPs': lf,
        'Param Reduction': f"{mp/lp:.2f}x", 'FLOPs Reduction': f"{mf/lf:.2f}x",
    })
complexity_theory_df = pd.DataFrame(rows)
print(f"\nTable: Theoretical complexity, single attention layer, E={E_val}, H={H_val}, R={R_val}\n")
print(complexity_theory_df.to_string(index=False))


In [ ]:

attn_lora = LowRankSparseMultiheadAttention(embed_dim=768, num_heads=GLOBAL_NUM_HEADS, rank=GLOBAL_RANK, sparsity_ratio=0.5).cuda()
attn_mha  = StandardMHAWrapper(embed_dim=768, num_heads=GLOBAL_NUM_HEADS).cuda()

dummy_tokens = torch.randn(1, 9, 768).cuda()

flops_lora, params_lora = profile(attn_lora, inputs=(dummy_tokens,), verbose=False)
flops_mha,  params_mha  = profile(attn_mha,  inputs=(dummy_tokens,), verbose=False)

time_lora = measure_inference_time(attn_lora, dummy_tokens, n_runs=100)
time_mha  = measure_inference_time(attn_mha, dummy_tokens, n_runs=100)
mem_lora  = measure_memory(attn_lora, dummy_tokens)
mem_mha   = measure_memory(attn_mha, dummy_tokens)

empirical_complexity_df = pd.DataFrame({
    'Standard MHA': {'Params': int(params_mha), 'FLOPs': int(flops_mha),
                      'Time (ms)': round(time_mha, 4), 'Peak Memory (MB)': round(mem_mha, 4)},
    'LoRa-SMHA':    {'Params': int(params_lora), 'FLOPs': int(flops_lora),
                      'Time (ms)': round(time_lora, 4), 'Peak Memory (MB)': round(mem_lora, 4)},
}).T
print(f"Table: Empirical (thop-profiled) complexity, isolated attention layer, N=9, E=768, H={GLOBAL_NUM_HEADS}, R={GLOBAL_RANK}\n")
print(empirical_complexity_df.to_string())


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

x = np.arange(2)
width = 0.35
axes[0].bar(x - width/2, [params_mha, params_lora], width, label='Params')
axes[0].bar(x + width/2, [flops_mha, flops_lora], width, label='FLOPs')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Standard MHA', 'LoRa-SMHA'])
axes[0].set_yscale('log')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Attention-layer Params & FLOPs (N=9, E=768)')
axes[0].legend()

N_range = np.array([9, 16, 25, 49, 100, 196])
mha_flops_vals = [int(mha_flops.subs({E: 768, N: int(n)})) for n in N_range]
lora_flops_vals = [int(lora_flops.subs({E: 768, N: int(n), R: GLOBAL_RANK})) for n in N_range]
axes[1].plot(N_range, mha_flops_vals, marker='o', label='Standard MHA')
axes[1].plot(N_range, lora_flops_vals, marker='s', label=f'LoRa-SMHA (R={GLOBAL_RANK})')
axes[1].axvline(9, color='gray', linestyle='--', alpha=0.5)
axes[1].text(9, max(mha_flops_vals)*0.9, 'this paper\n(grid=3)', fontsize=8, ha='left')
axes[1].set_xlabel('N (tokens)')
axes[1].set_ylabel('FLOPs')
axes[1].set_title('Theoretical FLOPs vs Sequence Length')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/complexity_analysis.png', dpi=150)
plt.show()


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out.detach()

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())
        self.model.zero_grad()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def remove(self):
        self._fwd.remove()
        self._bwd.remove()


In [ ]:

class FullRankSparseAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, sparsity_ratio=0.5):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.sparsity_ratio = sparsity_ratio
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.scale = embed_dim ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        b, h, t, _ = attn_scores.size()
        if t == 1:
            return attn_scores
        k = max(1, int(sparsity_ratio * t))
        top, _ = torch.topk(attn_scores, k=k, dim=-1)
        thresh = top.min(dim=-1, keepdim=True)[0]
        mask = attn_scores >= thresh
        return attn_scores.masked_fill(~mask, float('-inf'))

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        scores = self.sparse_attention(scores, self.sparsity_ratio)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)

class SingleBackboneHybridModel(nn.Module):
    def __init__(self, num_classes, backbone='resnet', embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, sparsity_ratio=0.5, grid_size=3, drop_path_rate=0.1):
        super().__init__()
        assert backbone in ('resnet', 'densenet')
        if backbone == 'resnet':
            self.backbone = ResNet18_Features()
            in_channels = 512
        else:
            self.backbone = DenseNet121_Features()
            in_channels = 1024
        self.grid_size = grid_size
        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(in_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        if self.grid_size != feats.shape[-1]:
            feats = F.adaptive_avg_pool2d(feats, (self.grid_size, self.grid_size))
        b, c, h, w = feats.shape
        feats = feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)


### 3.2 Pathology foundation models (UNI / CONCH / Virchow) — R3 fix merged in

This replaces the original Section 18 encoder-loading cell with the **fixed** version from the R3 standalone notebook: CONCH is loaded via its own `conch` package (not `transformers.AutoModel`), and Virchow's raw token-sequence output is pooled into the documented 2560-dim embedding. One-time install needed (internet required):

In [ ]:
!pip install -q git+https://github.com/Mahmoodlab/CONCH.git

In [ ]:
FOUNDATION_MODEL_EPOCHS = GLOBAL_NUM_EPOCHS

class _CONCHImageEncoderWrapper(nn.Module):
    def __init__(self, conch_model):
        super().__init__()
        self.conch_model = conch_model

    def forward(self, x):
        return self.conch_model.encode_image(x, proj_contrast=False, normalize=False)

class _VirchowEncoderWrapper(nn.Module):
    def __init__(self, vit_model):
        super().__init__()
        self.vit_model = vit_model

    def forward(self, x):
        output = self.vit_model(x)
        class_token = output[:, 0]
        patch_tokens = output[:, 1:]
        return torch.cat([class_token, patch_tokens.mean(1)], dim=-1)

def load_foundation_encoder(name):
    if name == 'UNI':
        import timm
        encoder = timm.create_model(
            "hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, num_classes=0
        )
        embed_dim = 1024
    elif name == 'CONCH':

        from conch.open_clip_custom import create_model_from_pretrained
        conch_model, _ = create_model_from_pretrained(
            'conch_ViT-B-16', "hf_hub:MahmoodLab/conch", hf_auth_token=globals().get('hf_token')
        )
        encoder = _CONCHImageEncoderWrapper(conch_model)
        embed_dim = 512
    elif name == 'Virchow':
        import timm
        vit_model = timm.create_model(
            "hf-hub:paige-ai/Virchow", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU
        )
        encoder = _VirchowEncoderWrapper(vit_model)
        embed_dim = 2560
    elif name == 'HIPT':

        raise NotImplementedError("Point this at your local HIPT ViT-S/16 checkpoint (see comment above).")
    else:
        raise ValueError(f"Unknown foundation model: {name}")

    encoder = encoder.cuda().eval()
    for p in encoder.parameters():
        p.requires_grad = False
    return encoder, embed_dim

class LinearProbeHead(nn.Module):
    def __init__(self, encoder, embed_dim, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            feats = self.encoder(x)
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
        return self.head(feats)

class CLAMAttentionHead(nn.Module):
    def __init__(self, encoder, embed_dim, num_classes, attn_dim=256):
        super().__init__()
        self.encoder = encoder
        self.attn_V = nn.Linear(embed_dim, attn_dim)
        self.attn_U = nn.Linear(embed_dim, attn_dim)
        self.attn_w = nn.Linear(attn_dim, 1)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            feats = self.encoder(x)
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
        a = torch.tanh(self.attn_V(feats)) * torch.sigmoid(self.attn_U(feats))
        gate = torch.sigmoid(self.attn_w(a))
        gated_feats = feats * gate
        return self.classifier(gated_feats)

Hugging Face login (gated models) — run once, before the dataset loop:

In [ ]:

from huggingface_hub import login
import getpass

hf_login_ok = False
hf_token = None

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except ModuleNotFoundError:
    pass
except Exception as e:
    print(f"Kaggle Secrets lookup failed ({type(e).__name__}: {e}) -- falling back.")

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    try:
        from huggingface_hub import HfApi
        HfApi().whoami()
        hf_login_ok = True
        print("Reusing existing cached Hugging Face login (from huggingface-cli login).")
    except Exception:
        pass

if not hf_login_ok and not hf_token:
    print("No cached login, HF_TOKEN env var, or Kaggle secret found.")
    print("Paste your Hugging Face token below (from https://huggingface.co/settings/tokens)")
    print("and press Enter -- your typing will be hidden.")
    hf_token = getpass.getpass("HF token: ")

if not hf_login_ok:
    try:
        login(token=hf_token)
        hf_login_ok = True
        print("Hugging Face login successful.")
    except Exception as e:
        print(f"Hugging Face login FAILED: {type(e).__name__}: {e}")
        print("Foundation model downloads (UNI/CONCH/Virchow) below will be skipped until this is fixed --")
        print("see the setup steps in this cell's comments.")


In [ ]:

from huggingface_hub import HfApi

api = HfApi()

if not hf_login_ok:
    print("Not logged in -- fix the cell above first (check your HF_TOKEN Kaggle Secret).")
else:
    try:
        who = api.whoami()
        print(f"Logged in as: {who['name']}  (email verified: {who.get('email', 'unknown')})\n")
    except Exception as e:
        print(f"Logged in, but whoami() failed: {type(e).__name__}: {e}\n")

    gated_repos = ['MahmoodLab/uni', 'MahmoodLab/CONCH', 'paige-ai/Virchow']
    print("Gated model access status:\n")
    for repo_id in gated_repos:
        try:
            api.model_info(repo_id)
            print(f"  [ACCESS GRANTED]  {repo_id}")
        except Exception as e:
            reason = type(e).__name__
            if 'Gated' in reason or '403' in str(e):
                print(f"  [PENDING/DENIED]  {repo_id}  -- request not yet approved (or was denied)")
            else:
                print(f"  [ERROR]           {repo_id}  -- {reason}: {e}")
    print("\nOnce all three show [ACCESS GRANTED], Section 18 below will download and run for real.")


## 4. `run_pipeline` — the entire per-dataset pipeline (Sections 2, 8–22)

Calling `run_pipeline("CRC7k", DATASET_CONFIGS["CRC7k"]["path"])` runs data loading, teacher-model loading, the main LoRaS-CT vs MHA-Net comparison, every results table/plot, t-SNE, LIME, SHAP, Grad-CAM, the statistical-significance / ablation / foundation-model / k-fold / hyperparameter-ablation / teacher-finetuning / patient-split-audit sections — all labeled with that dataset's name, with images saved under `images/CRC7k/...`.

It returns a dict of the key result tables for that dataset.

In [ ]:
def run_pipeline(CURRENT_DATASET, DATASET_CFG):
    is_presplit = "test_path" in DATASET_CFG
    _path_desc = (f"train_val={DATASET_CFG['train_val_path']} | test={DATASET_CFG['test_path']}"
                  if is_presplit else DATASET_CFG["path"])
    print(f"\n{'#'*80}\n# RUNNING PIPELINE FOR DATASET: {CURRENT_DATASET}\n# Path: {_path_desc}\n{'#'*80}\n")
    img_dir = f"images/{CURRENT_DATASET}"
    os.makedirs(img_dir, exist_ok=True)

    if is_presplit:
        dataset_path = DATASET_CFG["train_val_path"]
        train_val_dataset = datasets.ImageFolder(DATASET_CFG["train_val_path"])
        test_dataset_full = datasets.ImageFolder(DATASET_CFG["test_path"])
        assert train_val_dataset.classes == test_dataset_full.classes, (
            f"[{CURRENT_DATASET}] Class mismatch between train/val and test folders: "
            f"{train_val_dataset.classes} vs {test_dataset_full.classes}. "
            f"Both folders must contain identically-named class sub-folders."
        )
        dataset = train_val_dataset
        num_classes = len(dataset.classes)
    else:
        dataset_path = DATASET_CFG["path"]
        dataset = datasets.ImageFolder(dataset_path)
        num_classes = len(dataset.classes)

    _split_cache_path = f'{OUTPUT_ROOT}/{CURRENT_DATASET}_split_cache.json'

    if os.path.exists(_split_cache_path):
        print(f"[{CURRENT_DATASET}] Found cached split ({_split_cache_path}) -- reusing the exact "
              f"same train/val/test files as the run that first created it.")
        with open(_split_cache_path) as _f:
            _cache = json.load(_f)
        _train_root = DATASET_CFG["train_val_path"] if is_presplit else dataset_path
        _test_root = DATASET_CFG["test_path"] if is_presplit else dataset_path
        train_dataset = _rebuild_subset_from_paths(_train_root, _cache['train_paths'], train_val_transform)
        val_dataset = _rebuild_subset_from_paths(_train_root, _cache['val_paths'], train_val_transform)
        test_dataset = _rebuild_subset_from_paths(_test_root, _cache['test_paths'], test_transform)
    else:
        if is_presplit:
            val_size = int(len(train_val_dataset) * 0.1)
            train_size = len(train_val_dataset) - val_size
            split_generator = torch.Generator().manual_seed(42)
            train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size], generator=split_generator)

            train_dataset.dataset.transform = train_val_transform
            val_dataset.dataset.transform = train_val_transform
            test_dataset_full.transform = test_transform
            test_dataset = test_dataset_full
        else:
            group_split = patient_group_split(dataset, seed=42)
            if group_split is not None:
                train_idx, val_idx, test_idx = group_split
                print(f"[{CURRENT_DATASET}] Patient/slide IDs detected -- using patient-level "
                      f"GroupShuffleSplit (no patient appears in more than one split).")

                train_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
                val_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
                test_dataset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
            else:
                test_size = int(len(dataset) * 0.2)
                train_size = len(dataset) - test_size
                split_generator = torch.Generator().manual_seed(42)
                train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=split_generator)

                val_size = int(train_size * 0.1)
                train_size = train_size - val_size
                train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size], generator=split_generator)

                train_dataset.dataset.transform = train_val_transform
                val_dataset.dataset.transform = train_val_transform
                test_dataset.dataset.transform = test_transform

        _cache = {
            'train_paths': _flatten_subset_to_paths(train_dataset),
            'val_paths': _flatten_subset_to_paths(val_dataset),
            'test_paths': _flatten_subset_to_paths(test_dataset),
        }
        with open(_split_cache_path, 'w') as _f:
            json.dump(_cache, _f)
        print(f"[{CURRENT_DATASET}] Split cache saved -- {_split_cache_path} "
              f"({len(_cache['train_paths'])} train / {len(_cache['val_paths'])} val / "
              f"{len(_cache['test_paths'])} test files)")

    train_dataset = quick_subset(train_dataset, QUICK_TEST_MAX_TRAIN)
    val_dataset = quick_subset(val_dataset, QUICK_TEST_MAX_VAL)
    test_dataset = quick_subset(test_dataset, QUICK_TEST_MAX_TEST)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    print(f"[{CURRENT_DATASET}] Classes: {dataset.classes}")
    print(f"[{CURRENT_DATASET}] Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}"
          + ("  [QUICK_TEST_MODE: capped]" if QUICK_TEST_MODE else ""))

    teacher_vit = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_deit = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_swin = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()

    teacher_models = [teacher_vit, teacher_deit, teacher_swin]

    torch.save(teacher_vit.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_vit.pth')
    torch.save(teacher_deit.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_deit.pth')
    torch.save(teacher_swin.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_swin.pth')

    os.system(f"ls -la {OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_*.pth 2>/dev/null || echo 'no checkpoints yet'")

    def run_full_comparison(num_classes, teacher_models, train_loader, val_loader, test_loader,
                             num_epochs=20, device='cuda', alpha=0.5, temperature=3.0,
                             lr=0.01, momentum=0.9, grid_size=3):

        models_to_compare = {

            'MHA-Net (baseline)': MHANetBaseline(num_classes, num_heads=8, num_layers=2, grid_size=grid_size).to(device),
            'LoRaS-CT': HybridStudentModel(num_classes, grid_size=grid_size).to(device),
        }

        results = {}
        trained_models = {}

        for name, model in models_to_compare.items():
            print(f"\n{'='*60}")
            print(f"Training: {name}  (grid_size={grid_size} -> N={grid_size*grid_size} tokens)")
            print(f"{'='*60}")

            cached = load_model_checkpoint(CURRENT_DATASET, name, model)
            if cached is not None:
                results[name], cached_history = cached
                trained_models[name] = {'model': model, **cached_history}
                continue

            criterion = DistillationLoss(alpha=alpha, temperature=temperature)
            optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

            trained_model, train_losses, val_losses, train_accs, val_accs = \
                train_model_with_distillation(
                    model, teacher_models, train_loader, val_loader,
                    criterion, optimizer, num_epochs=num_epochs
                )

            test_accuracy, test_loss = evaluate_model(trained_model, test_loader, criterion.ce_loss)
            print(f"[{name}] Test Accuracy: {test_accuracy:.2f}%, Test Loss: {test_loss:.4f}")

            dummy_input = torch.randn(1, 3, 224, 224).to(device)
            params = count_parameters(trained_model)
            flops, _ = profile(trained_model, inputs=(dummy_input,), verbose=False)
            inf_time = measure_inference_time(trained_model, dummy_input, device=device)
            memory = measure_memory(trained_model, dummy_input, device=device)

            results[name] = {
                'Test Accuracy (%)': test_accuracy,
                'Params (M)': params / 1e6,
                'FLOPs (G)': flops / 1e9,
                'Inference Time (ms)': inf_time,
                'Peak Memory (MB)': memory,
            }
            history = {'train_losses': train_losses, 'val_losses': val_losses,
                       'train_accs': train_accs, 'val_accs': val_accs}
            trained_models[name] = {'model': trained_model, **history}

            save_model_checkpoint(CURRENT_DATASET, name, trained_model, results[name], history)

        print(f"\n{'='*80}")
        print(f"FINAL COMPARISON -- {CURRENT_DATASET}")
        print(f"{'='*80}")
        metrics = ['Test Accuracy (%)', 'Params (M)', 'FLOPs (G)', 'Inference Time (ms)', 'Peak Memory (MB)']
        header = f"{'Metric':<25}" + "".join(f"{name:<25}" for name in results.keys())
        print(header)
        for metric in metrics:
            row = f"{metric:<25}"
            for name in results.keys():
                row += f"{results[name][metric]:<25.4f}"
            print(row)

        return results, trained_models

    comparison_results, comparison_trained_models = run_full_comparison(
        num_classes=num_classes,
        teacher_models=teacher_models,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        num_epochs=GLOBAL_NUM_EPOCHS,
        device='cuda',
        grid_size=3,
    )

    loras_ct_model = comparison_trained_models['LoRaS-CT']['model']
    mha_net_model = comparison_trained_models['MHA-Net (baseline)']['model']

    print(f'LoRaS-CT (Algorithm-1-accurate) : {count_parameters(loras_ct_model):,} parameters')
    print(f'MHA-Net (baseline)         : {count_parameters(mha_net_model):,} parameters')
    print(f'Teacher ViT                : {count_parameters(teacher_vit):,} parameters')
    print(f'Teacher DeiT                : {count_parameters(teacher_deit):,} parameters')
    print(f'Teacher Swin Transformer   : {count_parameters(teacher_swin):,} parameters')

  

    _eff_embed_dim, _eff_num_heads, _eff_num_layers, _eff_grid_size = 768, GLOBAL_NUM_HEADS, GLOBAL_NUM_LAYERS, 3
    _eff_N = _eff_grid_size * _eff_grid_size
    _eff_dummy = torch.randn(1, 3, 224, 224).to('cuda')

    
    mha_net_model.to('cpu')
    loras_ct_model.to('cpu')
    torch.cuda.empty_cache()
    gc.collect()

    _efficiency_models = {
        'MHA-Net (baseline)': mha_net_model,
        'LoRaS-CT': loras_ct_model,
        'Linformer': GenericHybridModel(
            num_classes, attn_factory=lambda: LinformerAttention(_eff_embed_dim, _eff_num_heads, seq_len=_eff_N),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
        'Performer': GenericHybridModel(
            num_classes, attn_factory=lambda: PerformerAttention(_eff_embed_dim, _eff_num_heads),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
        'BigBird': GenericHybridModel(
            num_classes, attn_factory=lambda: BigBirdAttention(_eff_embed_dim, _eff_num_heads, block_size=64),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
    }

    print(f"\n{'='*70}\nSection 8b: isolated efficiency measurement -- {CURRENT_DATASET}\n{'='*70}")
    isolated_efficiency_table = measure_efficiency_isolated(
        _efficiency_models, _eff_dummy, device='cuda', n_runs=100
    )
    print(isolated_efficiency_table.to_string())

    
    for _mname in ['MHA-Net (baseline)', 'LoRaS-CT']:
        comparison_results[_mname]['Inference Time (ms)'] = isolated_efficiency_table.loc[_mname, 'Inference Time (ms)']
        comparison_results[_mname]['Peak Memory (MB)'] = isolated_efficiency_table.loc[_mname, 'Peak Memory (MB)']

    
    del _efficiency_models
    torch.cuda.empty_cache()
    gc.collect()

    
    mha_net_model.to('cuda')
    loras_ct_model.to('cuda')

    def get_predictions_and_labels(model, data_loader, input_size=None):

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images, labels = images.cuda(), labels.cuda()
                if input_size is not None:
                    images = F.interpolate(images, size=input_size, mode='bilinear', align_corners=False)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.array(all_preds), np.array(all_labels)

    def get_predictions_and_labels_probs(model, data_loader):
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images, labels = images.cuda(), labels.cuda()
                outputs = model(images)
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        return np.vstack(all_preds), np.concatenate(all_labels)

    def plot_confusion_matrix(y_true, y_pred, class_names, save_path="images/confusion_matrix.png"):
       
        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

        plt.figure(figsize=(6, 6), dpi=300)
        disp.plot(cmap=plt.cm.Blues, ax=plt.gca(), values_format='d', colorbar=False)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                plt.text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=10, color='black',
                         fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.4'))
        plt.xticks(rotation=45, ha='right', fontsize=12)
        plt.yticks(fontsize=12)
        plt.xlabel('Predicted Labels', fontsize=12)
        plt.ylabel('True Labels', fontsize=12)
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()

       
        all_labels = list(range(len(class_names)))
        print("Accuracy:", accuracy_score(y_true, y_pred))
        print("Precision:", precision_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("Recall:", recall_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("F1 Score:", f1_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("\nClassification Report:\n", classification_report(
            y_true, y_pred, labels=all_labels, target_names=class_names, zero_division=0))

    def plot_roc_curve(y_true, y_scores, class_names, save_path="images/roc_curve.png"):
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
        
        if n_classes == 2 and y_true_bin.shape[1] == 1:
            y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
        plt.figure(figsize=(6, 5), dpi=300)
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=3, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc='lower right', fontsize=8)
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()

    def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies,
                              save_path="images/training_curves.png"):
        epochs = range(1, len(train_losses) + 1)
        plt.figure(figsize=(10, 4), dpi=300)

        plt.subplot(1, 2, 1)
        plt.plot(epochs, train_losses, label='Train Loss', color='blue', lw=3)
        plt.plot(epochs, val_losses, label='Validation Loss', color='orange', lw=3)
        plt.title('Training and Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend(fontsize=10)

        plt.subplot(1, 2, 2)
        plt.plot(epochs, train_accuracies, label='Train Accuracy', color='green', lw=3)
        plt.plot(epochs, val_accuracies, label='Validation Accuracy', color='red', lw=3)
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy (%)')
        plt.legend(fontsize=10)

        plt.tight_layout()
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()

    model_to_evaluate = loras_ct_model
    model_to_evaluate_name = 'LoRaS-CT'
    train_losses_to_plot = comparison_trained_models[model_to_evaluate_name]['train_losses']
    val_losses_to_plot = comparison_trained_models[model_to_evaluate_name]['val_losses']
    train_accs_to_plot = comparison_trained_models[model_to_evaluate_name]['train_accs']
    val_accs_to_plot = comparison_trained_models[model_to_evaluate_name]['val_accs']

    test_predictions, test_labels = get_predictions_and_labels(model_to_evaluate, test_loader)
    plot_confusion_matrix(test_labels, test_predictions, dataset.classes,
                           save_path=f"{img_dir}/confusion_matrix.png")

    test_probs, test_labels_probs = get_predictions_and_labels_probs(model_to_evaluate, test_loader)
    plot_roc_curve(test_labels_probs, test_probs, dataset.classes,
                    save_path=f"{img_dir}/roc_curve.png")

    plot_training_curves(train_losses_to_plot, val_losses_to_plot, train_accs_to_plot, val_accs_to_plot,
                          save_path=f"{img_dir}/training_curves.png")

    print(f"Peak Memory Usage (MB): {comparison_results[model_to_evaluate_name]['Peak Memory (MB)']:.2f}")
    print(f"Per-image Inference Time (ms): {comparison_results[model_to_evaluate_name]['Inference Time (ms)']:.4f}")

    print(f"Number of training images: {len(train_loader.dataset)}")
    print(f"Number of validation images: {len(val_loader.dataset)}")
    print(f"Number of test images: {len(test_loader.dataset)}")

    if hasattr(model_to_evaluate, 'resnet') and hasattr(model_to_evaluate, 'densenet'):
        custom_params = count_custom_parameters(model_to_evaluate, exclude_list=[model_to_evaluate.resnet, model_to_evaluate.densenet])
        print(f'Number of trainable parameters in custom (non-backbone) layers: {custom_params}')

    embed_dim, num_heads, num_layers = 768, GLOBAL_NUM_HEADS, GLOBAL_NUM_LAYERS
    grid_size = 3
    N = grid_size * grid_size
    rank = GLOBAL_RANK

    def count_params(module):
        return sum(p.numel() for p in module.parameters() if p.requires_grad)

    attn_factories = {
        'Linformer':               lambda: LinformerAttention(embed_dim, num_heads, seq_len=N),
        'Performer':                lambda: PerformerAttention(embed_dim, num_heads),
        'BigBird':                  lambda: BigBirdAttention(embed_dim, num_heads, block_size=64),
        'Standard MHA (MHA-Net)':   lambda: StandardMHAWrapper(embed_dim, num_heads),
        'LoRa-SMHA (proposed)':     lambda: LowRankSparseMultiheadAttention(embed_dim, num_heads, rank=rank),
    }

    dummy_img = torch.randn(1, 3, 224, 224)
    dummy_seq = torch.randn(1, N, embed_dim)

    table10_rows = {}
    for name, factory in attn_factories.items():
        attn_module = factory()
        attn_params = count_params(attn_module)
        attn_flops, _ = profile(attn_module, inputs=(dummy_seq,), verbose=False)

        full_model = GenericHybridModel(
            num_classes, attn_factory=factory, embed_dim=embed_dim,
            num_layers=num_layers, grid_size=grid_size
        )
        full_params = count_params(full_model)
        full_flops, _ = profile(full_model, inputs=(dummy_img,), verbose=False)

        table10_rows[name] = {
            'Attn. Params': attn_params,
            'Attn. FLOPs': attn_flops,
            'Full-Model Params': full_params,
            'Full-Model FLOPs': full_flops,
        }

    table10 = pd.DataFrame(table10_rows).T[
        ['Attn. Params', 'Attn. FLOPs', 'Full-Model Params', 'Full-Model FLOPs']
    ]

    _table10_name_map = {
        'Standard MHA (MHA-Net)': 'MHA-Net (baseline)',
        'LoRa-SMHA (proposed)': 'LoRaS-CT',
        'Linformer': 'Linformer',
        'Performer': 'Performer',
        'BigBird': 'BigBird',
    }
    table10['Inference Time (ms)'] = [
        isolated_efficiency_table.loc[_table10_name_map[n], 'Inference Time (ms)'] for n in table10.index
    ]
    table10['Peak Memory (MB)'] = [
        isolated_efficiency_table.loc[_table10_name_map[n], 'Peak Memory (MB)'] for n in table10.index
    ]

    baselines = ['Linformer', 'Performer', 'BigBird', 'Standard MHA (MHA-Net)']
    best_attn = table10.loc[baselines, 'Attn. Params'].min()
    best_flops = table10.loc[baselines, 'Attn. FLOPs'].min()
    best_full_p = table10.loc[baselines, 'Full-Model Params'].min()
    best_full_f = table10.loc[baselines, 'Full-Model FLOPs'].min()

    proposed = table10.loc['LoRa-SMHA (proposed)']
    reduction = pd.Series({
        'Attn. Params':       f"{(1 - proposed['Attn. Params'] / best_attn) * 100:.1f}%",
        'Attn. FLOPs':        f"{(1 - proposed['Attn. FLOPs'] / best_flops) * 100:.1f}%",
        'Full-Model Params':  f"{(1 - proposed['Full-Model Params'] / best_full_p) * 100:.1f}%",
        'Full-Model FLOPs':   f"{(1 - proposed['Full-Model FLOPs'] / best_full_f) * 100:.1f}%",
    }, name='Reduction vs. best baseline')

    print(f"Table 10: attention-mechanism efficiency comparison -- {CURRENT_DATASET} (N={N} tokens, embed_dim={embed_dim}, heads={num_heads})\n")
    print(table10.round(0).to_string())
    print()
    print(reduction.to_string())

    print(f"\n{'='*60}\nSection 9c: Trained baseline + SOTA comparison [{CURRENT_DATASET}]\n{'='*60}")

    def _compute_prf(labels, preds):
        return (
            precision_score(labels, preds, average='weighted', zero_division=0),
            recall_score(labels, preds, average='weighted', zero_division=0),
            f1_score(labels, preds, average='weighted', zero_division=0),
        )

    sota_comparison_rows = {}

  
    _lora_prec, _lora_rec, _lora_f1 = _compute_prf(test_labels, test_predictions)
    sota_comparison_rows['LoRaS-CT'] = {
        'Accuracy (%)': round(comparison_results['LoRaS-CT']['Test Accuracy (%)'], 2),
        'Precision': round(_lora_prec, 4),
        'Recall': round(_lora_rec, 4),
        'F1-score': round(_lora_f1, 4),
    }

  

    _mha_preds, _mha_labels = get_predictions_and_labels(mha_net_model, test_loader)
    _mha_prec, _mha_rec, _mha_f1 = _compute_prf(_mha_labels, _mha_preds)
    sota_comparison_rows['MHA-Net (Standard MHA)'] = {
        'Accuracy (%)': round(comparison_results['MHA-Net (baseline)']['Test Accuracy (%)'], 2),
        'Precision': round(_mha_prec, 4),
        'Recall': round(_mha_rec, 4),
        'F1-score': round(_mha_f1, 4),
    }
    print(f"[MHA-Net (Standard MHA)] reused from Section 8 -- "
          f"Acc={sota_comparison_rows['MHA-Net (Standard MHA)']['Accuracy (%)']:.2f}%  "
          f"F1={sota_comparison_rows['MHA-Net (Standard MHA)']['F1-score']:.4f}")

    attn_baseline_factories_trained = {
        'Linformer':               lambda: LinformerAttention(embed_dim, num_heads, seq_len=N),
        'Performer':                lambda: PerformerAttention(embed_dim, num_heads),
        'BigBird':                  lambda: BigBirdAttention(embed_dim, num_heads, block_size=64),
    }

    for _name, _factory in attn_baseline_factories_trained.items():
        print(f"\n--- Training attention baseline: {_name} [{CURRENT_DATASET}] ---")
        _m = GenericHybridModel(num_classes, attn_factory=_factory, embed_dim=embed_dim,
                                 num_layers=num_layers, grid_size=grid_size).cuda()
        _crit = DistillationLoss(alpha=0.5, temperature=3.0)
        _opt = optim.SGD(_m.parameters(), lr=0.01, momentum=0.9)
        train_model_with_distillation(_m, teacher_models, train_loader, val_loader,
                                       _crit, _opt, num_epochs=GLOBAL_NUM_EPOCHS)
        _acc, _ = evaluate_model(_m, test_loader, _crit.ce_loss)
        _preds, _labels = get_predictions_and_labels(_m, test_loader)
        _prec, _rec, _f1 = _compute_prf(_labels, _preds)
        sota_comparison_rows[_name] = {
            'Accuracy (%)': round(_acc, 2), 'Precision': round(_prec, 4),
            'Recall': round(_rec, 4), 'F1-score': round(_f1, 4),
        }
        torch.save(_m.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_attn_baseline_{_name.replace(" ", "_")}.pth')
        _m.cpu(); torch.cuda.empty_cache()

   
    SOTA_BACKBONES = {
        'DeiT B16':  'deit_base_patch16_224',
        'ViT B16':   'vit_base_patch16_224',
        'ViT L16':   'vit_large_patch16_224',
        'ViT L32':   'vit_large_patch32_224',
        'ViT B32':   'vit_base_patch32_224',
        'Swin B4':   'swin_base_patch4_window7_224',
        'Swin V2 S': 'swinv2_small_window16_256',
        'Swin L4':   'swin_large_patch4_window7_224',
    }
    if QUICK_TEST_MODE:
      
        SOTA_BACKBONES = dict(list(SOTA_BACKBONES.items())[:2])
        print("QUICK_TEST_MODE: only smoke-testing 2/8 SOTA backbones.")

    def _train_plain_classifier(model, train_loader, val_loader, num_epochs, lr=1e-4, input_size=None):
        
        model = model.cuda()
        optimizer = optim.AdamW(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        for _epoch in range(num_epochs):
            model.train()
            for images, labels in train_loader:
                images, labels = images.cuda(), labels.cuda()
                if input_size is not None:
                    images = F.interpolate(images, size=input_size, mode='bilinear', align_corners=False)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()
        return model

    for _display_name, _timm_name in SOTA_BACKBONES.items():
        print(f"\n--- Fine-tuning SOTA backbone: {_display_name} ({_timm_name}) [{CURRENT_DATASET}] ---")
        try:
            _sota_model = create_model(_timm_name, pretrained=True, num_classes=num_classes)
        except Exception as _e:
            print(f"  Skipping {_display_name}: could not load '{_timm_name}' ({_e})")
            continue
        
        _backbone_input_size = 256 if _timm_name == 'swinv2_small_window16_256' else None
        try:
            _sota_model = _train_plain_classifier(_sota_model, train_loader, val_loader,
                                                    GLOBAL_NUM_EPOCHS, input_size=_backbone_input_size)
            _preds, _labels = get_predictions_and_labels(_sota_model, test_loader,
                                                            input_size=_backbone_input_size)
            _acc = accuracy_score(_labels, _preds) * 100
            _prec, _rec, _f1 = _compute_prf(_labels, _preds)
            sota_comparison_rows[_display_name] = {
                'Accuracy (%)': round(_acc, 2), 'Precision': round(_prec, 4),
                'Recall': round(_rec, 4), 'F1-score': round(_f1, 4),
            }
            torch.save(_sota_model.state_dict(),
                       f'{OUTPUT_ROOT}/{CURRENT_DATASET}_sota_backbone_{_display_name.replace(" ", "_")}.pth')
        except Exception as _e:
            print(f"  Skipping {_display_name}: {_e}")
        finally:
            _sota_model.cpu(); del _sota_model; torch.cuda.empty_cache()

    sota_comparison_df = pd.DataFrame(sota_comparison_rows).T[
        ['Accuracy (%)', 'Precision', 'Recall', 'F1-score']
    ]
    print(f"\nSection 9c comparison table -- {CURRENT_DATASET} "
          f"(efficient-attention baselines + SOTA backbones vs LoRaS-CT, reused)\n")
    print(sota_comparison_df.to_string())

    _fig, _ax = plt.subplots(figsize=(10, 6))
    _model_order = list(sota_comparison_df.index)
    _ax.plot(_model_order, sota_comparison_df['Precision'], marker='o', label='Precision')
    _ax.plot(_model_order, sota_comparison_df['Recall'], marker='s', linestyle='--', label='Recall')
    _ax.set_xlabel('Models'); _ax.set_ylabel('Score')
    _ax.set_title(f'Precision and Recall across baseline models -- {CURRENT_DATASET}')
    plt.xticks(rotation=45, ha='right')
    _ax.legend(); plt.tight_layout()
    plt.savefig(f"{img_dir}/sota_precision_recall_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

    def extract_pooled_features(model, data_loader):
        model.eval()
        all_features, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images = images.cuda()
                features = model(images)
                features = features.view(features.size(0), -1)
                all_features.append(features.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.vstack(all_features), np.array(all_labels)

    def plot_tsne(features, labels, title, class_names):

        n_samples = features.shape[0]
        perplexity = min(30, max(n_samples - 1, 1))
        tsne = TSNE(n_components=2, random_state=0, perplexity=perplexity)
        features_2d = tsne.fit_transform(features)

        plt.figure(figsize=(6, 6))
        scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels,
                               cmap=plt.get_cmap('tab10', len(class_names)), alpha=0.7)
        plt.xlabel('t-SNE Component 1')
        plt.ylabel('t-SNE Component 2')
        plt.title(title)
        plt.legend(handles=scatter.legend_elements()[0], labels=class_names, loc='best')
        plt.savefig(f'{img_dir}/{title.lower().replace(" ", "_")}.png', dpi=300)
        plt.show()

    class ResNet18_PooledFeatures(nn.Module):
        def __init__(self):
            super().__init__()
            resnet = models.resnet18(pretrained=True)
            self.model = nn.Sequential(*list(resnet.children())[:-1])

        def forward(self, x):
            return self.model(x)

    class DenseNet121_PooledFeatures(nn.Module):
        def __init__(self):
            super().__init__()
            densenet = models.densenet121(pretrained=True)
            self.features = densenet.features

        def forward(self, x):
            x = self.features(x)
            x = F.relu(x, inplace=False)
            x = F.adaptive_avg_pool2d(x, (1, 1))
            return x

    resnet_pooled_model = ResNet18_PooledFeatures().cuda()
    densenet_pooled_model = DenseNet121_PooledFeatures().cuda()

    resnet_tsne_features, resnet_tsne_labels = extract_pooled_features(resnet_pooled_model, test_loader)
    densenet_tsne_features, densenet_tsne_labels = extract_pooled_features(densenet_pooled_model, test_loader)
    combined_tsne_features = np.concatenate((resnet_tsne_features, densenet_tsne_features), axis=1)

    print("ResNet features shape:", resnet_tsne_features.shape)
    print("DenseNet features shape:", densenet_tsne_features.shape)
    print("Combined features shape:", combined_tsne_features.shape)

    plot_tsne(resnet_tsne_features, resnet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of ResNet18 Features', dataset.classes)
    plot_tsne(densenet_tsne_features, densenet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of DenseNet121 Features', dataset.classes)
    plot_tsne(combined_tsne_features, resnet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of Combined Features', dataset.classes)

    def extract_hybrid_features_for_tsne(model, data_loader):
        model.eval()
        all_features, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images = images.cuda()
                _, feats = model(images, return_features=True)
                feats = feats.view(feats.size(0), -1)
                all_features.append(feats.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.vstack(all_features), np.array(all_labels)

    hybrid_tsne_features, hybrid_tsne_labels = extract_hybrid_features_for_tsne(loras_ct_model, test_loader)

    n_samples = hybrid_tsne_features.shape[0]
    perplexity = min(30, max(n_samples - 1, 1))
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    hybrid_tsne_2d = tsne.fit_transform(hybrid_tsne_features)

    df_tsne = pd.DataFrame(data=hybrid_tsne_2d, columns=['Dim1', 'Dim2'])
    df_tsne['Label'] = hybrid_tsne_labels

    plt.figure(figsize=(10, 8))
    sns.scatterplot(x='Dim1', y='Dim2', hue='Label', data=df_tsne, palette='tab10', marker='o', alpha=0.7)
    plt.title(f'{CURRENT_DATASET} - t-SNE Visualization of Hybrid Model Features')
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.legend(title='Classes', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.savefig(f'{img_dir}/hybrid_tsne.png', dpi=300, bbox_inches='tight')
    plt.show()

    from lime import lime_image
    from skimage.segmentation import mark_boundaries

    def lime_predict_fn(images):
        images_tensor = torch.tensor(images).permute(0, 3, 1, 2).float().cuda()
        with torch.no_grad():
            outputs = model_to_evaluate(images_tensor)
        return F.softmax(outputs, dim=1).cpu().numpy()

    lime_explainer = lime_image.LimeImageExplainer()

    lime_image_idx = 0
    lime_test_image = test_loader.dataset[lime_image_idx][0].unsqueeze(0)
    lime_test_image_np = lime_test_image.squeeze(0).cpu().numpy().transpose(1, 2, 0)

    lime_explanation = lime_explainer.explain_instance(
        lime_test_image_np,
        lime_predict_fn,
        top_labels=5,
        hide_color=0,
        num_samples=1000,
    )

    lime_label_to_explain = lime_explanation.top_labels[0]
    lime_temp, lime_mask = lime_explanation.get_image_and_mask(
        lime_label_to_explain, positive_only=True, num_features=10, hide_rest=False
    )

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(lime_test_image_np)
    plt.title(f"{CURRENT_DATASET} - Original Image")

    plt.subplot(1, 2, 2)
    plt.imshow(mark_boundaries(lime_temp, lime_mask))
    plt.title(f"{CURRENT_DATASET} - LIME Explanation for Class: {lime_label_to_explain}")
    plt.show()

 
    import shap

    shap_image_idx = 0
    shap_test_image, _ = test_loader.dataset[shap_image_idx]
    shap_test_image = shap_test_image.unsqueeze(0)
    print("Shape of the image tensor before conversion:", shap_test_image.shape)

   
    shap_n_background = min(50, len(test_loader.dataset))
    shap_background = torch.stack([test_loader.dataset[i][0] for i in range(shap_n_background)]).cuda()

    model_to_evaluate.eval()

    shap_explainer = shap.GradientExplainer(model_to_evaluate, shap_background)

    
    shap_values, indexes = shap_explainer.shap_values(
        shap_test_image.cuda(), ranked_outputs=1
    )
    shap_values_class = shap_values[0]

    shap_map = np.mean(np.abs(shap_values_class), axis=0)
    shap_map = (shap_map - shap_map.min()) / (shap_map.max() - shap_map.min() + 1e-8)
    if shap_map.ndim == 3:
        shap_map = shap_map.mean(axis=0)

    shap_display_image = shap_test_image.squeeze(0).permute(1, 2, 0).cpu().numpy()
    shap_display_image = np.clip(shap_display_image, 0, 1)

    predicted_class_idx = int(indexes[0][0])
    predicted_class_name = (
        dataset.classes[predicted_class_idx] if "dataset" in globals() else predicted_class_idx
    )

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(shap_display_image)
    plt.axis('off')
    plt.title(f'{CURRENT_DATASET} - Original Image')

    plt.subplot(1, 2, 2)
    plt.imshow(shap_display_image)
    plt.imshow(shap_map, cmap='jet', alpha=0.5)
    plt.axis('off')
    plt.title(f'SHAP Explanation (class: {predicted_class_name})')
    plt.show()

   
    def compute_metrics_table(models_dict, data_loader, comparison_results):
        rows = {}
        for name, model in models_dict.items():
            preds, labels = get_predictions_and_labels(model, data_loader)
            rows[name] = {
                'Accuracy':  comparison_results[name]['Test Accuracy (%)'] / 100,
                'Precision': precision_score(labels, preds, average='weighted', zero_division=0),
                'Recall':    recall_score(labels, preds, average='weighted', zero_division=0),
                'F1-score':  f1_score(labels, preds, average='weighted', zero_division=0),
            }
        df = pd.DataFrame(rows).T[['Accuracy', 'Precision', 'Recall', 'F1-score']]
        return df.round(4)

    models_dict = {
        'LoRaS-CT': loras_ct_model,
        'MHA-Net (baseline)': mha_net_model,
    }

    metrics_table = compute_metrics_table(models_dict, test_loader, comparison_results)
    print(f"Table 3: Performance metrics on {CURRENT_DATASET} (Accuracy matches Section 8 FINAL COMPARISON exactly)\n")
    print(metrics_table.to_string())

  
    efficiency_table = pd.DataFrame(comparison_results).T[
        ['Params (M)', 'FLOPs (G)', 'Inference Time (ms)', 'Peak Memory (MB)', 'Test Accuracy (%)']
    ].round(4)
    print(f"Table 5: Computational efficiency on {CURRENT_DATASET} (identical to Section 8 FINAL COMPARISON)\n")
    print(efficiency_table.to_string())

    def count_all_parameters(module):
        return sum(p.numel() for p in module.parameters())

    def param_breakdown(models_dict):
        rows = {}
        backbone_params = None
        for name, model in models_dict.items():
            total_trainable = count_parameters(model)
            if hasattr(model, 'resnet') and hasattr(model, 'densenet'):
                head_only = count_custom_parameters(model, exclude_list=[model.resnet, model.densenet])
                if backbone_params is None:
                    backbone_params = count_all_parameters(model.resnet) + count_all_parameters(model.densenet)
            else:
                head_only = total_trainable
            rows[name] = {
                'Total trainable params': f'{total_trainable:,}',
                'Transformer+head only (excl. CNN backbones)': f'{head_only:,}',
            }
        df = pd.DataFrame(rows).T
        return df, backbone_params

    param_table, shared_backbone_params = param_breakdown(models_dict)
    print(param_table.to_string())
    print(f"\nShared CNN backbone params (ResNet18+DenseNet121 features, same for both models): {shared_backbone_params:,}")

    print(f"\n[{CURRENT_DATASET}] Consistency check vs. Section 8 FINAL COMPARISON (Params (M)):")
    for name in models_dict:
        total_trainable = count_parameters(models_dict[name])
        reported = comparison_results[name]['Params (M)'] * 1e6
        status = "OK" if abs(total_trainable - reported) < 1 else "MISMATCH -- investigate"
        print(f"  {name}: param_breakdown={total_trainable:,}  vs  comparison_results={reported:,.0f}  [{status}]")

    model_builders = {
        'LoRaS-CT': lambda: HybridStudentModel(num_classes, grid_size=3).cuda(),
        'MHA-Net (baseline)': lambda: MHANetBaseline(num_classes, grid_size=3).cuda(),
    }

    multiseed_results = {name: {'accuracy': [], 'f1': [], 'precision': [], 'recall': []} for name in model_builders}

    for seed in SEED_LIST:
        print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")
        set_all_seeds(seed)
        seed_train_loader, seed_val_loader, seed_test_loader = make_split(dataset_path, seed)

        for name, builder in model_builders.items():
            set_all_seeds(seed)
            model = builder()
            model_name = f"{name}_seed{seed}"

            cached = load_model_checkpoint(CURRENT_DATASET, model_name, model)
            if cached is not None:
                cached_result, _ = cached
                acc, f1, prec, rec = (cached_result['accuracy'], cached_result['f1'],
                                       cached_result['precision'], cached_result['recall'])
            else:
                criterion = DistillationLoss(alpha=0.5, temperature=3.0)
                optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

                model, *_ = train_model_with_distillation(
                    model, teacher_models, seed_train_loader, seed_val_loader,
                    criterion, optimizer, num_epochs=STAT_NUM_EPOCHS
                )

                acc, _, y_true, y_pred = evaluate_model(model, seed_test_loader, criterion.ce_loss, return_predictions=True)
                f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
                prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
                rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)

                save_model_checkpoint(CURRENT_DATASET, model_name, model,
                                       {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec}, {})

            multiseed_results[name]['accuracy'].append(acc)
            multiseed_results[name]['f1'].append(f1)
            multiseed_results[name]['precision'].append(prec)
            multiseed_results[name]['recall'].append(rec)

            print(f"[seed {seed}] {name}: Acc={acc:.2f}%  F1={f1:.4f}")

            del model
            torch.cuda.empty_cache()
            gc.collect()

    summary_rows = {}
    for name, m in multiseed_results.items():
        summary_rows[name] = {
            'Accuracy (%)': f"{np.mean(m['accuracy']):.2f} +/- {np.std(m['accuracy']):.2f}",
            'F1-score':     f"{np.mean(m['f1']):.4f} +/- {np.std(m['f1']):.4f}",
            'Precision':    f"{np.mean(m['precision']):.4f} +/- {np.std(m['precision']):.4f}",
            'Recall':       f"{np.mean(m['recall']):.4f} +/- {np.std(m['recall']):.4f}",
            'N runs':       len(m['accuracy']),
        }
    stat_summary_df = pd.DataFrame(summary_rows).T
    print(f"Table: Mean +/- SD over {N_SEEDS} independent seeds -- {CURRENT_DATASET}\n")
    print(stat_summary_df.to_string())

    acc_a = multiseed_results['LoRaS-CT']['accuracy']
    acc_b = multiseed_results['MHA-Net (baseline)']['accuracy']
    if N_SEEDS < 2:
        print(f"\nPaired t-test skipped: N_SEEDS={N_SEEDS} (need >= 2 seeds to estimate variance).")
        print("  This is expected under QUICK_TEST_MODE. Set QUICK_TEST_MODE = False for a real test.")
    else:
        t_stat, p_value = stats.ttest_rel(acc_a, acc_b)
        print(f"\nPaired t-test (LoRaS-CT vs MHA-Net, accuracy, N={N_SEEDS} seeds):")
        print(f"  t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
        print(f"  {'Statistically significant at alpha=0.05' if p_value < 0.05 else 'NOT statistically significant at alpha=0.05'}")
        print("  NOTE: with N_SEEDS=3 this test has very low statistical power -- treat the p-value")
        print("  as indicative only. Increase N_SEEDS to 5-10 for the camera-ready submission.")

    fig, ax = plt.subplots(figsize=(6, 4))
    names = list(multiseed_results.keys())
    means = [np.mean(multiseed_results[n]['accuracy']) for n in names]
    stds = [np.std(multiseed_results[n]['accuracy']) for n in names]
    colors = ['#2E86AB', '#A23B72']
    ax.bar(names, means, yerr=stds, capsize=8, color=colors[:len(names)])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Test Accuracy Mean +/- SD over {N_SEEDS} Seeds')
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + s + 0.3, f'{m:.2f}+/-{s:.2f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/stat_significance_barplot.png', dpi=150)
    plt.show()

    gradcam_target_layer = model_to_evaluate.resnet.features[-1]
    gradcam = GradCAM(model_to_evaluate, gradcam_target_layer)

    n_examples = 4
    fig, axes = plt.subplots(2, n_examples, figsize=(4 * n_examples, 8))

    for i in range(n_examples):
        img_tensor, true_label = test_loader.dataset[i]
        input_tensor = img_tensor.unsqueeze(0).cuda()

        cam, pred_class = gradcam.generate(input_tensor)

        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

        true_name = dataset.classes[true_label] if "dataset" in globals() else true_label
        pred_name = dataset.classes[pred_class] if "dataset" in globals() else pred_class

        axes[0, i].imshow(img_np)
        axes[0, i].axis('off')
        axes[0, i].set_title(f"True: {true_name}", fontsize=9)

        axes[1, i].imshow(img_np)
        axes[1, i].imshow(cam, cmap='jet', alpha=0.5)
        axes[1, i].axis('off')
        axes[1, i].set_title(f"Grad-CAM (pred: {pred_name})", fontsize=9)

    plt.suptitle(f"Grad-CAM explanations -- {CURRENT_DATASET} ({model_to_evaluate_name})", y=1.02)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/gradcam_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

    gradcam.remove()

    ABLATION_EPOCHS = GLOBAL_NUM_EPOCHS

    def build_full_model():
        return HybridStudentModel(num_classes, grid_size=3).cuda()

    def build_no_lowrank():
        return GenericHybridModel(
            num_classes, attn_factory=lambda: FullRankSparseAttention(768, GLOBAL_NUM_HEADS, sparsity_ratio=0.5), grid_size=3
        ).cuda()

    def build_no_sparsity():
        return GenericHybridModel(
            num_classes, attn_factory=lambda: LowRankSparseMultiheadAttention(768, GLOBAL_NUM_HEADS, rank=GLOBAL_RANK, sparsity_ratio=1.0),
            grid_size=3
        ).cuda()

    def build_resnet_only():
        return SingleBackboneHybridModel(num_classes, backbone='resnet', grid_size=3).cuda()

    def build_densenet_only():
        return SingleBackboneHybridModel(num_classes, backbone='densenet', grid_size=3).cuda()

    ablation_configs = {
        'Full LoRaS-CT (all components)':    (build_full_model,    True),
        'w/o Low-Rank Factorization':        (build_no_lowrank,    True),
        'w/o Sparsity (top-k masking)':      (build_no_sparsity,   True),
        'w/o Knowledge Distillation':        (build_full_model,    False),
        'w/o DenseNet branch (ResNet only)': (build_resnet_only,   True),
        'w/o ResNet branch (DenseNet only)': (build_densenet_only, True),
    }

    ablation_results = {}

    ablation_results['Full LoRaS-CT (all components)'] = {
        'Accuracy (%)': comparison_results['LoRaS-CT']['Test Accuracy (%)'],
        'F1-score': metrics_table.loc['LoRaS-CT', 'F1-score'],
        'Params (M)': comparison_results['LoRaS-CT']['Params (M)'],
    }
    print(f"[Full LoRaS-CT (all components)] reused from Section 8 -- "
          f"Acc={ablation_results['Full LoRaS-CT (all components)']['Accuracy (%)']:.2f}%  "
          f"F1={ablation_results['Full LoRaS-CT (all components)']['F1-score']:.4f}  "
          f"Params={ablation_results['Full LoRaS-CT (all components)']['Params (M)']:.2f}M")

    for name, (builder, use_kd) in ablation_configs.items():
        if name == 'Full LoRaS-CT (all components)':
            continue
        print(f"\n{'='*60}\nAblation: {name}\n{'='*60}")
        set_all_seeds(42)
        model = builder()

        cached = load_model_checkpoint(CURRENT_DATASET, name, model)
        if cached is not None:
            cached_result, _ = cached
            acc = cached_result['Accuracy (%)']
            f1 = cached_result['F1-score']
            params_m = cached_result['Params (M)']
        else:
            if use_kd:
                criterion = DistillationLoss(alpha=0.5, temperature=3.0)
                optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
                model, *_ = train_model_with_distillation(
                    model, teacher_models, train_loader, val_loader, criterion, optimizer, num_epochs=ABLATION_EPOCHS
                )
                eval_criterion = criterion.ce_loss
            else:
                eval_criterion = nn.CrossEntropyLoss()
                optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
                train_model_plain(model, train_loader, val_loader, eval_criterion, optimizer, num_epochs=ABLATION_EPOCHS)

            acc, _, y_true, y_pred = evaluate_model(model, test_loader, eval_criterion, return_predictions=True)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            params_m = count_parameters(model) / 1e6

            save_model_checkpoint(CURRENT_DATASET, name, model,
                                   {'Accuracy (%)': acc, 'F1-score': f1, 'Params (M)': params_m}, {})

        ablation_results[name] = {'Accuracy (%)': acc, 'F1-score': f1, 'Params (M)': params_m}
        print(f"[{name}] Acc={acc:.2f}%  F1={f1:.4f}  Params={params_m:.2f}M")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    ablation_df = pd.DataFrame(ablation_results).T[['Accuracy (%)', 'F1-score', 'Params (M)']]
    full_acc = ablation_df.loc['Full LoRaS-CT (all components)', 'Accuracy (%)']
    ablation_df['Delta Accuracy (pp)'] = (ablation_df['Accuracy (%)'] - full_acc).round(2)
    print(f"Table: Component-wise ablation study -- {CURRENT_DATASET}\n")
    print(ablation_df.round(4).to_string())

    fig, ax = plt.subplots(figsize=(9, 5))
    names = list(ablation_results.keys())
    accs = [ablation_results[n]['Accuracy (%)'] for n in names]
    colors = ['#2E86AB'] + ['#A23B72'] * (len(names) - 1)
    bars = ax.barh(names, accs, color=colors)
    ax.axvline(full_acc, color='gray', linestyle='--', label='Full model accuracy')
    ax.set_xlabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Component-wise Ablation Study')
    ax.legend()
    for bar, acc in zip(bars, accs):
        ax.text(acc + 0.3, bar.get_y() + bar.get_height() / 2, f'{acc:.2f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/ablation_study.png', dpi=150)
    plt.show()

    foundation_models_to_test = ['UNI', 'CONCH', 'Virchow']
    if not globals().get('hf_login_ok', False):
        print("Hugging Face login was not successful (see the cell above) -- skipping all three")
        print("gated foundation models rather than hitting the same 401 three times.")
        foundation_models_to_test = []
    foundation_results = {}

    for fm_name in foundation_models_to_test:
        print(f"\n{'='*60}\nFoundation model: {fm_name}\n{'='*60}")
        try:
            encoder, embed_dim = load_foundation_encoder(fm_name)
            model = LinearProbeHead(encoder, embed_dim, num_classes).cuda()

            optimizer = optim.Adam(model.head.parameters(), lr=1e-3)
            criterion = nn.CrossEntropyLoss()
            train_model_plain(model, train_loader, val_loader, criterion, optimizer,
                               num_epochs=FOUNDATION_MODEL_EPOCHS)

            acc, _, y_true, y_pred = evaluate_model(model, test_loader, criterion, return_predictions=True)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            params = count_parameters(model)

            dummy_input = torch.randn(1, 3, 224, 224).cuda()
            inf_time = measure_inference_time(model, dummy_input)
            memory = measure_memory(model, dummy_input)

            foundation_results[fm_name] = {
                'Test Accuracy (%)': acc, 'F1-score': f1,
                'Trainable Params (M)': params / 1e6,
                'Total Params (M)': sum(p.numel() for p in model.parameters()) / 1e6,
                'Inference Time (ms)': inf_time, 'Peak Memory (MB)': memory,
            }
            print(f"[{fm_name}] Acc={acc:.2f}%  F1={f1:.4f}")

            torch.save(model.head.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_foundation_{fm_name}_head_only.pth')

            del model, encoder
            torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            print(f"[{fm_name}] SKIPPED -- {type(e).__name__}: {e}")
            print("  (Needs internet + HF gated-license access in this environment. Run "
                  "`huggingface-cli login` after accepting the license on the model's HF page.)")
            foundation_results[fm_name] = {
                'Test Accuracy (%)': np.nan, 'F1-score': np.nan,
                'Trainable Params (M)': np.nan, 'Total Params (M)': np.nan,
                'Inference Time (ms)': np.nan, 'Peak Memory (MB)': np.nan,
            }

    combined_results = dict(foundation_results)
    combined_results['LoRaS-CT (ours)'] = {
        'Test Accuracy (%)': ablation_results['Full LoRaS-CT (all components)']['Accuracy (%)']
            if 'ablation_results' in globals() else np.nan,
        'F1-score': ablation_results['Full LoRaS-CT (all components)']['F1-score']
            if 'ablation_results' in globals() else np.nan,
        'Trainable Params (M)': count_parameters(loras_ct_model) / 1e6,
        'Total Params (M)': sum(p.numel() for p in loras_ct_model.parameters()) / 1e6,
        'Inference Time (ms)': np.nan, 'Peak Memory (MB)': np.nan,
    }

    foundation_comparison_df = pd.DataFrame(combined_results).T
    print(f"Table: LoRaS-CT vs pathology foundation models (linear probe) -- {CURRENT_DATASET}\n")
    print(foundation_comparison_df.round(4).to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    valid = foundation_comparison_df.dropna(subset=['Test Accuracy (%)'])
    bar_colors = ['#F18F01'] * (len(valid) - 1) + ['#2E86AB'] if 'LoRaS-CT (ours)' in valid.index else ['#F18F01'] * len(valid)
    ax.bar(valid.index, valid['Test Accuracy (%)'], color=bar_colors)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'LoRaS-CT vs Pathology Foundation Models (linear probe, {CURRENT_DATASET})')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/foundation_model_comparison.png', dpi=150)
    plt.show()

    from sklearn.model_selection import KFold, GroupKFold
    from scipy import stats as scipy_stats

    K_FOLDS = 2 if QUICK_TEST_MODE else 5
    KFOLD_NUM_EPOCHS = GLOBAL_NUM_EPOCHS

    full_ds_for_kfold = datasets.ImageFolder(dataset_path)
    all_indices = np.arange(len(full_ds_for_kfold))
    kfold_groups = get_patient_groups(full_ds_for_kfold)

    if kfold_groups is not None:
        print(f"[{CURRENT_DATASET}] Patient/slide IDs detected -- using GroupKFold "
              f"(no patient split across folds).")
        kf = GroupKFold(n_splits=K_FOLDS)
        fold_iter = kf.split(all_indices, groups=kfold_groups)
    else:
        kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
        fold_iter = kf.split(all_indices)

    kfold_results = {name: [] for name in model_builders}

    for fold_idx, (trainval_idx, test_idx) in enumerate(fold_iter):
        print(f"\n{'#'*70}\n# FOLD {fold_idx + 1}/{K_FOLDS}\n{'#'*70}")
        set_all_seeds(42 + fold_idx)

        if kfold_groups is not None:
            trainval_groups = kfold_groups[trainval_idx]
            gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42 + fold_idx)
            tr_sub, val_sub = next(gss.split(trainval_idx, groups=trainval_groups))
            train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]
        else:
            n_val = int(len(trainval_idx) * 0.1)
            val_idx = trainval_idx[:n_val]
            train_idx = trainval_idx[n_val:]

        train_subset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
        val_subset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
        test_subset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
        train_subset = quick_subset(train_subset, QUICK_TEST_MAX_TRAIN)
        val_subset = quick_subset(val_subset, QUICK_TEST_MAX_VAL)
        test_subset = quick_subset(test_subset, QUICK_TEST_MAX_TEST)

        fold_train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
        fold_val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
        fold_test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=2)

        for name, builder in model_builders.items():
            set_all_seeds(42 + fold_idx)
            model = builder()
            model_name = f"{name}_fold{fold_idx}"

            cached = load_model_checkpoint(CURRENT_DATASET, model_name, model)
            if cached is not None:
                cached_result, _ = cached
                acc = cached_result['accuracy']
            else:
                criterion = DistillationLoss(alpha=0.5, temperature=3.0)
                optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
                model, *_ = train_model_with_distillation(
                    model, teacher_models, fold_train_loader, fold_val_loader, criterion, optimizer,
                    num_epochs=KFOLD_NUM_EPOCHS
                )
                acc, _ = evaluate_model(model, fold_test_loader, criterion.ce_loss)
                save_model_checkpoint(CURRENT_DATASET, model_name, model, {'accuracy': acc}, {})

            kfold_results[name].append(acc)
            print(f"[Fold {fold_idx + 1}] {name}: Test Acc = {acc:.2f}%")

            del model
            torch.cuda.empty_cache()
            gc.collect()

    def confidence_interval_95(values):
        values = np.array(values)
        mean = values.mean()
        if len(values) > 1:
            sem = scipy_stats.sem(values)
            ci = scipy_stats.t.interval(0.95, df=len(values) - 1, loc=mean, scale=sem)
        else:
            ci = (mean, mean)
        return mean, values.std(), ci

    kfold_summary_rows = {}
    for name, accs in kfold_results.items():
        mean, std, ci = confidence_interval_95(accs)
        kfold_summary_rows[name] = {
            'Mean Accuracy (%)': round(mean, 2),
            'Std Dev': round(std, 2),
            '95% CI Lower': round(ci[0], 2),
            '95% CI Upper': round(ci[1], 2),
            'K': len(accs),
        }
    kfold_summary_df = pd.DataFrame(kfold_summary_rows).T
    print(f"Table: {K_FOLDS}-Fold Cross-Validation Results with 95% Confidence Intervals -- {CURRENT_DATASET}\n")
    print(kfold_summary_df.to_string())

    if K_FOLDS < 2:
        print(f"\nPaired t-test skipped: K_FOLDS={K_FOLDS} (need >= 2 folds to estimate variance).")
    else:
        t_stat, p_value = scipy_stats.ttest_rel(kfold_results['LoRaS-CT'], kfold_results['MHA-Net (baseline)'])
        print(f"\nPaired t-test across {K_FOLDS} folds (LoRaS-CT vs MHA-Net): t={t_stat:.4f}, p={p_value:.4f}")
        print(f"  {'Statistically significant at alpha=0.05' if p_value < 0.05 else 'NOT statistically significant at alpha=0.05'}")

    fig, ax = plt.subplots(figsize=(6, 4))
    names = list(kfold_results.keys())
    means = [np.mean(kfold_results[n]) for n in names]
    cis = [confidence_interval_95(kfold_results[n])[2] for n in names]
    errs = np.array([[m - ci[0], ci[1] - m] for m, ci in zip(means, cis)]).T
    ax.bar(names, means, yerr=errs, capsize=8, color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: {K_FOLDS}-Fold CV Mean Accuracy with 95% CI')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/kfold_cv_results.png', dpi=150)
    plt.show()

    HPARAM_ABLATION_EPOCHS = GLOBAL_NUM_EPOCHS

    layer_configs = [1, 2] if QUICK_TEST_MODE else [1, 2, 3, 4]
    head_configs = [4, 8] if QUICK_TEST_MODE else [2, 4, 8, 16]

    hparam_results_layers = {}
    for nl in layer_configs:
        print(f"\n--- num_layers = {nl} ---")

        if nl == GLOBAL_NUM_LAYERS:
            acc = comparison_results['LoRaS-CT']['Test Accuracy (%)']
            params_m = comparison_results['LoRaS-CT']['Params (M)']
            hparam_results_layers[nl] = {'Test Accuracy (%)': acc, 'Params (M)': params_m}
            print(f"[num_layers={nl}] reused from Section 8 (LoRaS-CT default) -- "
                  f"Acc={acc:.2f}%  Params={params_m:.2f}M")
            continue

        set_all_seeds(42)
        model = HybridStudentModel(num_classes, num_layers=nl, grid_size=3).cuda()
        model_name = f"LoRaS-CT_layers{nl}"

        cached = load_model_checkpoint(CURRENT_DATASET, model_name, model)
        if cached is not None:
            cached_result, _ = cached
            acc, params_m = cached_result['Test Accuracy (%)'], cached_result['Params (M)']
        else:
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
            model, *_ = train_model_with_distillation(
                model, teacher_models, train_loader, val_loader, criterion, optimizer,
                num_epochs=HPARAM_ABLATION_EPOCHS
            )
            acc, _ = evaluate_model(model, test_loader, criterion.ce_loss)
            params_m = count_parameters(model) / 1e6
            save_model_checkpoint(CURRENT_DATASET, model_name, model,
                                   {'Test Accuracy (%)': acc, 'Params (M)': params_m}, {})

        hparam_results_layers[nl] = {'Test Accuracy (%)': acc, 'Params (M)': params_m}
        print(f"[num_layers={nl}] Acc={acc:.2f}%  Params={params_m:.2f}M")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    hparam_results_heads = {}
    for nh in head_configs:
        print(f"\n--- num_heads = {nh} ---")

        if nh == GLOBAL_NUM_HEADS:
            acc = comparison_results['LoRaS-CT']['Test Accuracy (%)']
            hparam_results_heads[nh] = {'Test Accuracy (%)': acc}
            print(f"[num_heads={nh}] reused from Section 8 (LoRaS-CT default) -- Acc={acc:.2f}%")
            continue

        set_all_seeds(42)
        model = HybridStudentModel(num_classes, num_heads=nh, grid_size=3).cuda()
        model_name = f"LoRaS-CT_heads{nh}"

        cached = load_model_checkpoint(CURRENT_DATASET, model_name, model)
        if cached is not None:
            cached_result, _ = cached
            acc = cached_result['Test Accuracy (%)']
        else:
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
            model, *_ = train_model_with_distillation(
                model, teacher_models, train_loader, val_loader, criterion, optimizer,
                num_epochs=HPARAM_ABLATION_EPOCHS
            )
            acc, _ = evaluate_model(model, test_loader, criterion.ce_loss)
            save_model_checkpoint(CURRENT_DATASET, model_name, model, {'Test Accuracy (%)': acc}, {})

        hparam_results_heads[nh] = {'Test Accuracy (%)': acc}
        print(f"[num_heads={nh}] Acc={acc:.2f}%")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    layers_df = pd.DataFrame(hparam_results_layers).T
    layers_df.index.name = 'num_layers'
    print(f"Table: Ablation over number of transformer layers -- {CURRENT_DATASET}\n")
    print(layers_df.round(4).to_string())

    heads_df = pd.DataFrame(hparam_results_heads).T
    heads_df.index.name = 'num_heads'
    print(f"\nTable: Ablation over number of attention heads -- {CURRENT_DATASET}\n")
    print(heads_df.round(4).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].plot(list(hparam_results_layers.keys()),
                 [v['Test Accuracy (%)'] for v in hparam_results_layers.values()],
                 marker='o', color='#2E86AB')
    axes[0].set_xlabel('Number of Transformer Layers')
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title(f'{CURRENT_DATASET}: Accuracy vs. Number of Layers')
    axes[0].set_xticks(layer_configs)

    axes[1].plot(list(hparam_results_heads.keys()),
                 [v['Test Accuracy (%)'] for v in hparam_results_heads.values()],
                 marker='s', color='#A23B72')
    axes[1].set_xlabel('Number of Attention Heads')
    axes[1].set_ylabel('Test Accuracy (%)')
    axes[1].set_title(f'{CURRENT_DATASET}: Accuracy vs. Number of Attention Heads')
    axes[1].set_xticks(head_configs)

    plt.tight_layout()
    plt.savefig(f'{img_dir}/hparam_ablation_layers_heads.png', dpi=150)
    plt.show()

    print("\nNote: the KD on/off ablation for this same reviewer comment is already reported")
    print("in Section 17 ('w/o Knowledge Distillation' row of the component-wise ablation table).")

    TEACHER_STAGE_EPOCHS = GLOBAL_NUM_EPOCHS
    HEAD_INIT_SEED = 42
    teacher_names = ['ViT-B/16', 'DeiT-B/16', 'Swin-B/4']
    _teacher_specs = [
        ('ViT-B/16', 'vit_base_patch16_224', 'teacher_vit'),
        ('DeiT-B/16', 'deit_base_patch16_224', 'teacher_deit'),
        ('Swin-B/4', 'swin_base_patch4_window7_224', 'teacher_swin'),
    ]

    def evaluate_teacher_standalone(teacher, data_loader):
        acc, _ = evaluate_model(teacher, data_loader, nn.CrossEntropyLoss())
        return acc

    def freeze_backbone_train_head_only(model):
        for pname, p in model.named_parameters():
            p.requires_grad = ('head' in pname)

    print(f"[{CURRENT_DATASET}] Evaluating each teacher across three INDEPENDENT stages -- "
          f"raw, linear probe, and full fine-tune. Each stage loads its own fresh, "
          f"separately-initialized teacher (same seed for the random head across stages, "
          f"but no warm-starting from one stage into the next).\n")

    pre_ft_accs, linear_probe_accs, post_ft_accs = {}, {}, {}

    for name, timm_name, stub in _teacher_specs:
        print(f"\n=== {name} ===")

        print(f"--- [A] Raw (ImageNet-pretrained backbone, brand-new untrained head -- "
              f"expect close to chance level) ---")
        set_all_seeds(HEAD_INIT_SEED)
        teacher_raw = create_model(timm_name, pretrained=True, num_classes=num_classes).cuda()
        acc_raw = evaluate_teacher_standalone(teacher_raw, test_loader)
        pre_ft_accs[name] = acc_raw
        print(f"  {name}: {acc_raw:.2f}%")
        torch.save(teacher_raw.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_{stub}_raw.pth')
        del teacher_raw
        torch.cuda.empty_cache()
        gc.collect()

        print(f"--- [B] Linear probe (separate fresh load, backbone frozen, only the head "
              f"trained for {TEACHER_STAGE_EPOCHS} epochs) ---")
        set_all_seeds(HEAD_INIT_SEED)
        teacher_probe = create_model(timm_name, pretrained=True, num_classes=num_classes).cuda()
        freeze_backbone_train_head_only(teacher_probe)
        head_params = [p for p in teacher_probe.parameters() if p.requires_grad]
        probe_optimizer = optim.Adam(head_params, lr=1e-3)
        probe_criterion = nn.CrossEntropyLoss()
        train_model_plain(teacher_probe, train_loader, val_loader, probe_criterion, probe_optimizer,
                           num_epochs=TEACHER_STAGE_EPOCHS)
        acc_probe = evaluate_teacher_standalone(teacher_probe, test_loader)
        linear_probe_accs[name] = acc_probe
        print(f"  {name}: {acc_probe:.2f}%")
        torch.save(teacher_probe.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_{stub}_linear_probe.pth')
        del teacher_probe
        torch.cuda.empty_cache()
        gc.collect()

        print(f"--- [C] Full fine-tune (separate fresh load, all layers unfrozen, "
              f"trained for {TEACHER_STAGE_EPOCHS} epochs) ---")
        set_all_seeds(HEAD_INIT_SEED)
        teacher_full = create_model(timm_name, pretrained=True, num_classes=num_classes).cuda()
        full_optimizer = optim.SGD(teacher_full.parameters(), lr=1e-4, momentum=0.9)
        full_criterion = nn.CrossEntropyLoss()
        train_model_plain(teacher_full, train_loader, val_loader, full_criterion, full_optimizer,
                           num_epochs=TEACHER_STAGE_EPOCHS)
        acc_full = evaluate_teacher_standalone(teacher_full, test_loader)
        post_ft_accs[name] = acc_full
        print(f"  {name}: {acc_full:.2f}%")
        torch.save(teacher_full.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_{stub}_finetuned.pth')
        del teacher_full
        torch.cuda.empty_cache()
        gc.collect()

    teacher_comparison_df = pd.DataFrame({
        'Before fine-tuning (%)': pre_ft_accs,
        'Linear probe (%)': linear_probe_accs,
        'After fine-tuning (%)': post_ft_accs,
    })
    print(f"\nTable: Teacher standalone performance, random head vs. linear probe vs. "
          f"full fine-tuning -- {CURRENT_DATASET}\n")
    print(teacher_comparison_df.round(2).to_string())

    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(teacher_names))
    width = 0.25
    ax.bar(x - width, list(pre_ft_accs.values()), width, label='Before fine-tuning (random head)', color='#F18F01')
    ax.bar(x, list(linear_probe_accs.values()), width, label='Linear probe (frozen backbone)', color='#6A994E')
    ax.bar(x + width, list(post_ft_accs.values()), width, label='After fine-tuning (full)', color='#2E86AB')
    ax.set_xticks(x)
    ax.set_xticklabels(teacher_names)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Teacher Standalone Accuracy -- Random Head vs. Linear Probe vs. Full Fine-tuning')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/teacher_finetuning_comparison.png', dpi=150)
    plt.show()

    class WeightedDistillationLoss(nn.Module):
        def __init__(self, num_teachers, alpha=0.5, temperature=3.0):
            super().__init__()
            self.alpha = alpha
            self.temperature = temperature
            self.ce_loss = nn.CrossEntropyLoss()
            self.kl_div = nn.KLDivLoss(reduction="batchmean")
            self.teacher_weights = nn.Parameter(torch.ones(num_teachers))

        def combine(self, teacher_logits_list):
            w = F.softmax(self.teacher_weights, dim=0)
            return sum(wi * tl for wi, tl in zip(w, teacher_logits_list))

        def forward(self, student_logits, teacher_logits_list, ground_truth):
            combined_teacher_logits = self.combine(teacher_logits_list)
            hard_loss = self.ce_loss(student_logits, ground_truth)
            soft_loss = self.kl_div(
                F.log_softmax(student_logits / self.temperature, dim=1),
                F.softmax(combined_teacher_logits / self.temperature, dim=1)
            ) * (self.temperature ** 2)
            return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

    def train_model_with_weighted_distillation(student_model, teacher_models, train_loader, val_loader,
                                                distillation_criterion, optimizer, num_epochs=1):
        for epoch in range(num_epochs):
            student_model.train()
            for images, labels in train_loader:
                images, labels = images.cuda(), labels.cuda()
                optimizer.zero_grad()
                student_outputs = student_model(images)
                with torch.no_grad():
                    teacher_logits_list = [teacher(images) for teacher in teacher_models]
                loss = distillation_criterion(student_outputs, teacher_logits_list, labels)
                loss.backward()
                optimizer.step()
            train_acc, _ = evaluate_model(student_model, train_loader, distillation_criterion.ce_loss)
            val_acc, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
            print(f'Epoch [{epoch+1}/{num_epochs}] Train Acc: {train_acc:.2f}%  Val Acc: {val_acc:.2f}%')
        return student_model

    KD_WEIGHTING_EPOCHS = GLOBAL_NUM_EPOCHS

    print(f"[{CURRENT_DATASET}] Building fresh (non-fine-tuned) teacher copies for the "
          f"simple-vs-weighted averaging comparison...")
    teacher_vit_kd = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_deit_kd = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_swin_kd = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_models_kd = [teacher_vit_kd, teacher_deit_kd, teacher_swin_kd]

    set_all_seeds(42)
    model_simple = HybridStudentModel(num_classes, grid_size=3).cuda()
    criterion_simple = DistillationLoss(alpha=0.5, temperature=3.0)
    optimizer_simple = optim.SGD(model_simple.parameters(), lr=0.01, momentum=0.9)
    model_simple, *_ = train_model_with_distillation(
        model_simple, teacher_models_kd, train_loader, val_loader, criterion_simple, optimizer_simple,
        num_epochs=KD_WEIGHTING_EPOCHS
    )
    acc_simple, _ = evaluate_model(model_simple, test_loader, criterion_simple.ce_loss)

    torch.save(model_simple.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_KD_simple_averaging.pth')

    del model_simple
    torch.cuda.empty_cache()
    gc.collect()

    set_all_seeds(42)
    model_weighted = HybridStudentModel(num_classes, grid_size=3).cuda()
    criterion_weighted = WeightedDistillationLoss(num_teachers=len(teacher_models_kd), alpha=0.5, temperature=3.0).cuda()
    optimizer_weighted = optim.SGD(
        list(model_weighted.parameters()) + list(criterion_weighted.parameters()), lr=0.01, momentum=0.9
    )
    model_weighted = train_model_with_weighted_distillation(
        model_weighted, teacher_models_kd, train_loader, val_loader, criterion_weighted, optimizer_weighted,
        num_epochs=KD_WEIGHTING_EPOCHS
    )
    acc_weighted, _ = evaluate_model(model_weighted, test_loader, criterion_weighted.ce_loss)

    torch.save(model_weighted.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_KD_weighted_averaging.pth')

    # teacher_models_kd was built fresh from pretrained (non-fine-tuned) weights above, specifically so
    # this simple-vs-weighted comparison never distills from the already-fine-tuned teacher_models. That
    # is unrelated to weight-saving and stays exactly as-is here -- these fresh teacher copies are
    # discarded (not saved) since they are identical, by construction, to a fresh
    # create_model(pretrained=True, ...) call and saving them would add nothing.
    del teacher_vit_kd, teacher_deit_kd, teacher_swin_kd, teacher_models_kd
    torch.cuda.empty_cache()
    gc.collect()

    learned_weights = F.softmax(criterion_weighted.teacher_weights.detach(), dim=0).cpu().numpy()

    kd_weighting_df = pd.DataFrame({
        'Simple averaging': {'Test Accuracy (%)': acc_simple},
        'Learned weighted averaging': {'Test Accuracy (%)': acc_weighted},
    }).T
    print(f"Table: Simple vs. learned-weighted teacher averaging in MTKD -- {CURRENT_DATASET}\n")
    print(kd_weighting_df.round(2).to_string())
    print(f"\nLearned teacher weights (softmax-normalized): "
          f"ViT={learned_weights[0]:.3f}  DeiT={learned_weights[1]:.3f}  Swin={learned_weights[2]:.3f}")

    del model_weighted
    torch.cuda.empty_cache()
    gc.collect()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Simple avg', 'Learned weighted'], [acc_simple, acc_weighted], color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: MTKD Simple vs. Weighted Teacher Averaging')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/kd_weighting_comparison.png', dpi=150)
    plt.show()

    import re

    def try_extract_patient_id(filename):
        patterns = [
            r'SOB_[A-Z]_[A-Z]+-(\d+-\d+)',
            r'^(P\d+)',
            r'patient[_-]?(\d+)',
            r'case[_-]?(\d+)',
        ]
        for pat in patterns:
            m = re.search(pat, filename, flags=re.IGNORECASE)
            if m:
                return m.group(1)
        return None

    def resolve_nested_subset_indices(subset):
        indices = list(subset.indices)
        base = subset.dataset
        while isinstance(base, torch.utils.data.Subset):
            indices = [base.indices[i] for i in indices]
            base = base.dataset
        return indices, base

    def audit_split_for_leakage(train_subset, val_subset, test_subset):
        train_idx, base_ds = resolve_nested_subset_indices(train_subset)
        val_idx, _ = resolve_nested_subset_indices(val_subset)
        test_idx, _ = resolve_nested_subset_indices(test_subset)

        filepaths = [s[0] for s in base_ds.samples]
        patient_ids = [try_extract_patient_id(os.path.basename(fp)) for fp in filepaths]
        n_matched = sum(1 for p in patient_ids if p is not None)

        print(f"Filenames with a recognizable patient/case ID pattern: {n_matched}/{len(filepaths)}")

        if n_matched == 0:
            print(f"\nNo patient/case ID could be parsed from filenames in the {CURRENT_DATASET} dataset folder.")
            print(f"For {CURRENT_DATASET}, this means no patient/slide metadata was recognized in the")
            print("filenames (or this dataset genuinely has no patient/slide grouping, e.g. a tile-level")
            print("texture benchmark like Kather5k/NCT100k). If that's expected for this dataset, state it")
            print("explicitly in the rebuttal rather than claiming a patient-level split it doesn't support.")
            print(f"If {CURRENT_DATASET} DOES have multiple patches per patient/slide (e.g. BreakHis),")
            print("extend try_extract_patient_id()'s regex patterns to match its actual filename")
            print("convention, then rerun this cell and verify zero overlap before reporting final numbers.")
            return None

        train_patients = set(patient_ids[i] for i in train_idx if patient_ids[i] is not None)
        val_patients = set(patient_ids[i] for i in val_idx if patient_ids[i] is not None)
        test_patients = set(patient_ids[i] for i in test_idx if patient_ids[i] is not None)

        overlap_train_test = train_patients & test_patients
        overlap_train_val = train_patients & val_patients
        overlap_val_test = val_patients & test_patients

        print(f"\nUnique patients -- train: {len(train_patients)}, val: {len(val_patients)}, test: {len(test_patients)}")
        print(f"Patient overlap train<->test: {len(overlap_train_test)}")
        print(f"Patient overlap train<->val:  {len(overlap_train_val)}")
        print(f"Patient overlap val<->test:   {len(overlap_val_test)}")

        if overlap_train_test or overlap_train_val or overlap_val_test:
            print("\nLEAKAGE DETECTED: at least one patient appears in more than one split.")
            print("ACTION NEEDED: re-split at the patient level (e.g. sklearn GroupShuffleSplit /")
            print("GroupKFold using patient_ids as the group key) before reporting final numbers.")
        else:
            print("\nNo patient overlap detected across splits (based on parsed IDs).")

        return {'train': train_patients, 'val': val_patients, 'test': test_patients}

    print(f"[{CURRENT_DATASET}] Auditing the split from Section 1 / Cell 7 (dataset_path = {dataset_path})\n")
    patient_audit_result = audit_split_for_leakage(train_dataset, val_dataset, test_dataset)

    dataset_results = {
        'comparison_results': comparison_results,
        'metrics_table_3': metrics_table,
        'efficiency_table_5': efficiency_table,
        'ablation_df': ablation_df,
        'stat_summary_df': stat_summary_df,
        'foundation_comparison_df': foundation_comparison_df,
        'kfold_summary_df': kfold_summary_df,
        'hparam_layers_df': layers_df,
        'hparam_heads_df': heads_df,
        'teacher_comparison_df': teacher_comparison_df,
        'kd_weighting_df': kd_weighting_df,
        'sota_comparison_df': sota_comparison_df,
        'patient_audit_result': patient_audit_result,
    }
    print(f"\n{'#'*80}\n# FINISHED PIPELINE FOR DATASET: {CURRENT_DATASET}\n{'#'*80}\n")
    return dataset_results


## 5. Driver — run the pipeline for the selected dataset(s)

Runs once per entry in `DATASETS_TO_RUN` (set in Section 2). Results for every dataset run are collected into `all_dataset_results`, keyed by dataset name.

In [ ]:
all_dataset_results = {}

for _ds_name in DATASETS_TO_RUN:
    _ds_cfg = DATASET_CONFIGS[_ds_name]

    _missing = _check_dataset_paths(_ds_cfg)
    if _missing:
        print(f"\n[{_ds_name}] SKIPPED -- path(s) not found: {_missing}")
        print(f"[{_ds_name}] Update DATASET_CONFIGS['{_ds_name}'] in the dataset-registry cell "
              f"above (or set DATA_ROOT) to point at your actual data location.\n")
        all_dataset_results[_ds_name] = {"error": f"path(s) not found: {_missing}"}
        continue

    try:
        all_dataset_results[_ds_name] = run_pipeline(_ds_name, _ds_cfg)
    except Exception as e:
        print(f"\n[{_ds_name}] PIPELINE FAILED -- {type(e).__name__}: {e}")
        print(f"[{_ds_name}] Skipping to the next dataset (if any).\n")
        all_dataset_results[_ds_name] = {"error": str(e)}
    finally:
        torch.cuda.empty_cache()
        gc.collect()

print(f"\n\nDone. Ran: {list(all_dataset_results.keys())}")


In [ ]:

for _ds_name, _res in all_dataset_results.items():
    print(f"\n=== {_ds_name} ===")
    if "error" in _res:
        print(f"  FAILED: {_res['error']}")
        continue
    print(_res["comparison_results"])


In [ ]:
import shutil
import hashlib
import sys
import platform
import json as _json
import timm

RELEASE_DIR = os.path.join(OUTPUT_ROOT, 'release')
os.makedirs(RELEASE_DIR, exist_ok=True)

HEADLINE_MODELS = ['LoRaS-CT', 'MHA-Net (baseline)']
TEACHER_STUBS = ['teacher_vit', 'teacher_deit', 'teacher_swin']

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    'implementation_details': {
        'global_num_epochs': GLOBAL_NUM_EPOCHS,
        'teacher_fine_tune_epochs': GLOBAL_NUM_EPOCHS,
        'linear_probe_epochs': 5,
        'grid_size': 3,
        'lora_rank': GLOBAL_RANK,
        'transformer_layers': GLOBAL_NUM_LAYERS,
        'transformer_heads': GLOBAL_NUM_HEADS,
        'batch_size': batch_size,
        'kd_optimizer': 'SGD(lr=0.01, momentum=0.9)',
        'kd_distillation': 'alpha=0.5, temperature=3.0',
        'teacher_fine_tune_optimizer': 'SGD(lr=1e-4, momentum=0.9)',
        'linear_probe_optimizer': 'Adam(lr=1e-3), backbone frozen',
        'seed': 42,
        'python_version': sys.version,
        'platform': platform.platform(),
        'torch_version': torch.__version__,
        'timm_version': timm.__version__,
        'cuda_available': torch.cuda.is_available(),
    },
    'checkpoints': [],
}

for _ds in DATASETS_TO_RUN:
    for _model in HEADLINE_MODELS:
        _safe = _model.replace(" ", "_").replace("(", "").replace(")", "")
        _src = f"{OUTPUT_ROOT}/{_ds}_{_safe}_checkpoint.pth"
        if os.path.exists(_src):
            _dst_name = f"{_ds}_{_safe}.pth"
            _dst = os.path.join(RELEASE_DIR, _dst_name)
            shutil.copy2(_src, _dst)
            manifest['checkpoints'].append({
                'dataset': _ds, 'model': _model, 'file': _dst_name,
                'format': 'dict with keys: model_state_dict, result, history',
                'size_mb': round(os.path.getsize(_dst) / (1024 ** 2), 2),
                'sha256': _sha256(_dst),
            })
    for _stub in TEACHER_STUBS:
        _src = f"{OUTPUT_ROOT}/{_ds}_{_stub}_finetuned.pth"
        if os.path.exists(_src):
            _dst_name = f"{_ds}_{_stub}_finetuned.pth"
            _dst = os.path.join(RELEASE_DIR, _dst_name)
            shutil.copy2(_src, _dst)
            manifest['checkpoints'].append({
                'dataset': _ds, 'model': _stub, 'file': _dst_name,
                'format': 'bare state_dict (torch.save(model.state_dict(), ...))',
                'size_mb': round(os.path.getsize(_dst) / (1024 ** 2), 2),
                'sha256': _sha256(_dst),
            })

with open(os.path.join(RELEASE_DIR, 'manifest.json'), 'w') as f:
    _json.dump(manifest, f, indent=2)

_readme = f"""# Trained weights -- release bundle

This bundle contains the trained weights for the headline models reported in the paper,
released in response to the reproducibility comment.

## Contents

- `{{dataset}}_LoRaS-CT.pth` -- the proposed model, trained via knowledge distillation.
- `{{dataset}}_MHA-Net_baseline.pth` -- the MHA-Net baseline, trained the same way.
- `{{dataset}}_teacher_{{vit,deit,swin}}_finetuned.pth` -- the three fine-tuned ViT-B/16 /
  DeiT-B/16 / Swin-B/4 teachers used to produce the distillation targets for the above.

## Loading

LoRaS-CT / MHA-Net checkpoints are a dict:
```python
ckpt = torch.load('{{dataset}}_LoRaS-CT.pth', map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
print(ckpt['result'])   # reported test accuracy / F1 / params at save time
```

Teacher checkpoints are a bare state_dict:
```python
model.load_state_dict(torch.load('{{dataset}}_teacher_vit_finetuned.pth', map_location='cpu'))
```

## Implementation details

See `manifest.json` -> `implementation_details` for every training hyperparameter
(epochs, optimizer, learning rate, batch size, seed, and the exact library/hardware
versions used to produce these weights), and `manifest.json` -> `checkpoints` for a
SHA-256 checksum of every file in this bundle.
"""
with open(os.path.join(RELEASE_DIR, 'README.md'), 'w') as f:
    f.write(_readme)

_zip_path = shutil.make_archive(os.path.join(OUTPUT_ROOT, 'release_bundle'), 'zip', RELEASE_DIR)
print(f"Release bundle created: {_zip_path}")
print(f"Contains {len(manifest['checkpoints'])} checkpoint files + manifest.json + README.md\n")
print(pd.DataFrame(manifest['checkpoints']).to_string())


In [ ]:
from datetime import datetime

_ckpt_rows = []
for _fname in sorted(os.listdir(OUTPUT_ROOT)):
    if not _fname.endswith('_checkpoint.pth'):
        continue
    _fpath = os.path.join(OUTPUT_ROOT, _fname)
    _size_mb = os.path.getsize(_fpath) / (1024 ** 2)
    _mtime = datetime.fromtimestamp(os.path.getmtime(_fpath)).strftime('%Y-%m-%d %H:%M:%S')
    _stem = _fname[:-len('_checkpoint.pth')]
    _ds_match = next((d for d in DATASET_CONFIGS if _stem.startswith(d + '_')), None)
    if _ds_match:
        _model_name = _stem[len(_ds_match) + 1:]
    else:
        _ds_match, _model_name = '(unknown)', _stem
    _ckpt_rows.append({'Dataset': _ds_match, 'Model': _model_name, 'Size (MB)': round(_size_mb, 2), 'Saved': _mtime})

_ckpt_df = pd.DataFrame(_ckpt_rows).sort_values(['Dataset', 'Model']).reset_index(drop=True)
print(f"Checkpoint directory: {OUTPUT_ROOT}\n")
print(f"Total checkpoints found: {len(_ckpt_df)}\n")
print(_ckpt_df.to_string())

print("\nPer-dataset checkpoint counts:")
print(_ckpt_df.groupby('Dataset').size().to_string())

print(f"\nTotal size on disk: {_ckpt_df['Size (MB)'].sum():.1f} MB")
